<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/research%2Fpost-e50-spider-level-anchor/67A_postE50_spider_level_anchor_validation_FULL_READY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 67A · Post-E50 — SPIDER Level Anchor Validation (Stage B: real SPIDER evidence)

**Corrección metodológica crítica (esta revisión):** SPIDER numera instancias vertebrales
(`1,2,3,...`), canal (`100`) y discos (`201,202,203,...`) de **inferior a superior**, pero eso es
**identidad relativa de instancia**, no nivel anatómico absoluto. `vertebra label=1` no es
necesariamente L5, y `disc label=201` no es necesariamente L5-S1. Este notebook **nunca** afirma
`L1-L2...L5-S1` a partir de SPIDER. SPIDER se usa exclusivamente para validar:

- disc instance detection
- disc centroid localization
- relative disc ordering
- número de discos visibles (comportamiento de FOV)
- consistencia multi-slice
- lógica de abstención

El naming anatómico absoluto queda `UNAVAILABLE_FROM_SPIDER_REFERENCE` salvo evidencia
independiente explícita -- nunca inferido por conteo, por `201...`, ni por `radiological_gradings`
sin validar semántica primero.

## Evidencia real ya obtenida en Colab (Stage B)

- PFI root: `/content/drive/MyDrive/PFI_MVP`
- SPIDER root: `/content/drive/MyDrive/PFI_MVP/data/SPIDER` — `images/images/*.mha`,
  `masks/masks/*.mha`, `overview.csv`, `radiological_gradings.csv`
- 447 images, 447 masks · overview: 447 series, 179 training, 39 validation, 218 pacientes totales,
  0 pacientes en más de un subset público · radiological_gradings: 1520 filas, 218 pacientes, 100%
  cobertura vs overview
- **TEST**: no existe en este dataset público local; el hidden SPIDER test queda fuera de scope y
  bloqueado (`TEST_SPLIT_LOCKED = True`, `public_test_availability = HIDDEN_EXTERNAL_NOT_AVAILABLE`)
- **Checkpoint (corregido esta revisión)**: el checkout git LOCAL de
  `models/final/sagittal_spider_multiclass_final_best.pt` **SÍ tiene el SHA correcto**
  (`cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944`), verificado directamente en
  este worktree. Lo que estaba desactualizado era una copia vieja en Drive
  (SHA `7dd393cc750311c98003516d8110136310c31e8b6f0f00b6815f949fd61ef15b`), no el historial git. El
  checkpoint se acepta **únicamente por SHA-256 calculado**, nunca por nombre de archivo ni ruta:
  en Colab se prioriza `PFI_POST_E50_SAGITTAL_CHECKPOINT` (aceptado solo si su SHA == `cf11...`);
  fuera de Colab se usa el checkout local del repo (GATE A = PASS si su SHA == `cf11...`). No se
  modifica ni reemplaza ningún checkpoint.

## Prohibiciones de esta revisión

NO se crea notebook nuevo, NO se crea rama nueva, NO commit, NO push, NO entrenar, NO modificar
Notebook 67, NO modificar checkpoints existentes, NO tocar test.

Rama: `research/post-e50-spider-level-anchor`


In [1]:
# --- Setup: environment detection ---
import hashlib
import io
import json
import os
import re
import subprocess
import sys
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

EXECUTION_START = time.time()

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)
print("ENVIRONMENT:", "COLAB" if IN_COLAB else "LOCAL")


IN_COLAB: True
ENVIRONMENT: COLAB


## STEP 1 — Mount Google Drive (Colab only)

No se monta Drive automáticamente sin interacción visible: solo si `IN_COLAB=True`, y
`drive.mount()` de por sí requiere autorización interactiva del usuario en Colab.


In [2]:
DRIVE_MOUNTED = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = Path('/content/drive/MyDrive').is_dir()
    print("Drive mounted:", DRIVE_MOUNTED)
else:
    print("Not in Colab: skipping Drive mount. Local execution uses environment variables instead.")


Mounted at /content/drive
Drive mounted: True


## STEP 1B — Runtime Post-E50 embebido y congelado

Para evitar depender del clon desactualizado `PFI_MVP/repo` de Drive **y también evitar requerir un
ZIP/runtime externo**, este notebook incluye solamente el subconjunto mínimo del runtime necesario
para 67A, copiado del commit inmutable de Notebook 67:

`0e97083d443225226beb1f705fd794578b1b17f9`

Fuentes de procedencia:

- `ai_service/pfi_ai_service/model_architectures.py`
- `ai_service/pfi_ai_service/real_inference_runtime.py`
- `ai_service/pfi_ai_service/settings.py`

No se modifica ningún modelo ni código productivo. El notebook sigue leyendo el checkpoint
`cf11...` desde Drive y lo acepta únicamente por SHA-256. Este bloque es una **copia congelada**,
no una nueva implementación -- ver `EMBEDDED_RUNTIME_SOURCE_COMMIT` / `EMBEDDED_RUNTIME_SCOPE`.


In [3]:
# --- Frozen runtime subset embedded from immutable Notebook-67 commit (NOT a new implementation) ---
from typing import Any, Mapping
import torch
from torch import nn
from PIL import Image

EMBEDDED_RUNTIME_SOURCE_COMMIT = "0e97083d443225226beb1f705fd794578b1b17f9"
EMBEDDED_RUNTIME_SCOPE = "minimal frozen sagittal inference helpers required by Notebook 67A"
EMBEDDED_RUNTIME_MODE = "embedded_frozen_subset"
EMBEDDED_RUNTIME_BOOTSTRAP_STATUS = "PASS"

EMBEDDED_RUNTIME_SOURCE_FILES = {
    "model_architectures.py_blob_sha": "62b3ccdd78c115892313355f580a421a10a85a29",
    "real_inference_runtime.py_blob_sha": "69d31f7542fdc99439b6d8da8a6dc9347c226bc1",
    "settings.py_blob_sha": "ca3cf89fc901c5e9ea9c5495956a978a11bc7851",
}

# ---- exact model architecture snapshot (commit 0e97083...) ----
class SagittalDoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SagittalUNet2D(nn.Module):
    # Arquitectura exacta del checkpoint sagital E5/E12.
    def __init__(self, in_channels: int = 1, num_classes: int = 4, base_channels: int = 16) -> None:
        super().__init__()
        self.enc1 = SagittalDoubleConv(in_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = SagittalDoubleConv(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = SagittalDoubleConv(base_channels * 2, base_channels * 4)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = SagittalDoubleConv(base_channels * 4, base_channels * 8)
        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=2, stride=2)
        self.dec3 = SagittalDoubleConv(base_channels * 8, base_channels * 4)
        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
        self.dec2 = SagittalDoubleConv(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=2, stride=2)
        self.dec1 = SagittalDoubleConv(base_channels * 2, base_channels)
        self.out_conv = nn.Conv2d(base_channels, num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        bottleneck = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.up3(bottleneck), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)


class AxialDoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class AxialUNet2D(nn.Module):
    # Arquitectura exacta del entrenamiento axial E10 (no usada por 67A, incluida por completitud del snapshot).
    def __init__(self, num_classes: int = 6, base_channels: int = 16) -> None:
        super().__init__()
        b = base_channels
        self.e1 = AxialDoubleConv(1, b)
        self.e2 = AxialDoubleConv(b, b * 2)
        self.e3 = AxialDoubleConv(b * 2, b * 4)
        self.pool = nn.MaxPool2d(2)
        self.mid = AxialDoubleConv(b * 4, b * 8)
        self.u3 = nn.ConvTranspose2d(b * 8, b * 4, 2, 2)
        self.d3 = AxialDoubleConv(b * 8, b * 4)
        self.u2 = nn.ConvTranspose2d(b * 4, b * 2, 2, 2)
        self.d2 = AxialDoubleConv(b * 4, b * 2)
        self.u1 = nn.ConvTranspose2d(b * 2, b, 2, 2)
        self.d1 = AxialDoubleConv(b * 2, b)
        self.out = nn.Conv2d(b, num_classes, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        mid = self.mid(self.pool(e3))
        x = self.d3(torch.cat([self.u3(mid), e3], dim=1))
        x = self.d2(torch.cat([self.u2(x), e2], dim=1))
        x = self.d1(torch.cat([self.u1(x), e1], dim=1))
        return self.out(x)


def checkpoint_state_dict(checkpoint: Any) -> Mapping[str, torch.Tensor]:
    if isinstance(checkpoint, Mapping):
        for key in ("model_state_dict", "state_dict", "model"):
            value = checkpoint.get(key)
            if isinstance(value, Mapping):
                return normalize_state_dict(value)
        if checkpoint and all(torch.is_tensor(value) for value in checkpoint.values()):
            return normalize_state_dict(checkpoint)
    raise ValueError("El checkpoint no contiene model_state_dict/state_dict utilizable")


def normalize_state_dict(state_dict: Mapping[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    normalized: dict[str, torch.Tensor] = {}
    for key, value in state_dict.items():
        clean_key = str(key)
        for prefix in ("module.", "model."):
            if clean_key.startswith(prefix):
                clean_key = clean_key[len(prefix):]
        normalized[clean_key] = value
    return normalized


def infer_base_channels(model_key: str, checkpoint: Any, state_dict: Mapping[str, torch.Tensor]) -> int:
    if isinstance(checkpoint, Mapping) and checkpoint.get("base_channels") is not None:
        return int(checkpoint["base_channels"])
    weight_key = "enc1.block.0.weight" if model_key == "sagittal_spider" else "e1.net.0.weight"
    weight = state_dict.get(weight_key)
    return int(weight.shape[0]) if weight is not None else 16


def infer_num_classes(model_key: str, checkpoint: Any, state_dict: Mapping[str, torch.Tensor]) -> int:
    if isinstance(checkpoint, Mapping) and checkpoint.get("num_classes") is not None:
        return int(checkpoint["num_classes"])
    weight_key = "out_conv.weight" if model_key == "sagittal_spider" else "out.weight"
    weight = state_dict.get(weight_key)
    fallback = 4 if model_key == "sagittal_spider" else 6
    return int(weight.shape[0]) if weight is not None else fallback


def build_checkpoint_model(model_key: str, checkpoint: Any) -> tuple[nn.Module, dict[str, Any]]:
    state_dict = checkpoint_state_dict(checkpoint)
    base_channels = infer_base_channels(model_key, checkpoint, state_dict)
    num_classes = infer_num_classes(model_key, checkpoint, state_dict)
    if model_key == "sagittal_spider":
        model: nn.Module = SagittalUNet2D(num_classes=num_classes, base_channels=base_channels)
    elif model_key == "axial_t2_alkafri":
        model = AxialUNet2D(num_classes=num_classes, base_channels=base_channels)
    else:
        raise KeyError(f"Arquitectura no registrada para model_key={model_key}")
    model.load_state_dict(state_dict, strict=True)
    target_size = (256, 256)
    if isinstance(checkpoint, Mapping) and checkpoint.get("target_size") is not None:
        raw_size = checkpoint["target_size"]
        target_size = (int(raw_size[0]), int(raw_size[1]))
    return model, {
        "baseChannels": base_channels,
        "numClasses": num_classes,
        "targetSize": target_size,
        "checkpointKeys": sorted(str(key) for key in checkpoint.keys()) if isinstance(checkpoint, Mapping) else [],
    }


# ---- exact inference utility subset (commit 0e97083...) ----
def robust_percentile_normalize(array: np.ndarray, p_low: float = 1.0, p_high: float = 99.0) -> np.ndarray:
    value = np.asarray(array, dtype=np.float32)
    finite = np.isfinite(value)
    if not finite.any():
        return np.zeros_like(value, dtype=np.float32)
    low, high = np.percentile(value[finite], [p_low, p_high])
    if float(high) <= float(low):
        return np.zeros_like(value, dtype=np.float32)
    clipped = np.clip(value, low, high)
    return ((clipped - low) / (float(high) - float(low) + 1e-8)).astype(np.float32)


def resize_image(array: np.ndarray, target_size: tuple[int, int]) -> np.ndarray:
    normalized = robust_percentile_normalize(array)
    image = Image.fromarray(np.clip(normalized * 255.0, 0, 255).astype(np.uint8))
    resized = image.resize((target_size[1], target_size[0]), resample=Image.Resampling.BILINEAR)
    return np.asarray(resized, dtype=np.float32) / 255.0


LUMBAR_DISC_LEVELS = ("L1-L2", "L2-L3", "L3-L4", "L4-L5", "L5-S1")


def connected_instances(binary: np.ndarray, min_pixels: int = 20) -> list[np.ndarray]:
    # Separa una mascara de clase en sus componentes conexas, de superior a inferior.
    try:
        import SimpleITK as sitk
    except Exception:
        return []
    labelled = sitk.GetArrayFromImage(
        sitk.ConnectedComponent(sitk.GetImageFromArray(binary.astype(np.uint8)))
    )
    instances = [
        component
        for value in sorted(int(item) for item in np.unique(labelled) if int(item) != 0)
        if int((component := labelled == value).sum()) >= min_pixels
    ]
    instances.sort(key=lambda mask: float(np.where(mask)[0].mean()))
    return instances


MODEL_REGISTRY = {
    "sagittal_spider": {
        "plane": "sagittal",
        "num_classes": 4,
        "class_names": {
            0: "background",
            1: "vertebra_group",
            2: "canal",
            3: "disc_group",
        },
        "human_review_required": True,
    },
    "axial_t2_alkafri": {
        "plane": "axial",
        "num_classes": 6,
        "class_names": {
            0: "background_250",
            1: "raw_0",
            2: "raw_50",
            3: "raw_100",
            4: "raw_150",
            5: "raw_200",
        },
        "human_review_required": True,
    },
}

print("Embedded Post-E50 runtime subset:", EMBEDDED_RUNTIME_BOOTSTRAP_STATUS)
print("EMBEDDED_RUNTIME_SOURCE_COMMIT:", EMBEDDED_RUNTIME_SOURCE_COMMIT)
print("EMBEDDED_RUNTIME_SCOPE:", EMBEDDED_RUNTIME_SCOPE)
print(json.dumps(EMBEDDED_RUNTIME_SOURCE_FILES, indent=2))


Embedded Post-E50 runtime subset: PASS
EMBEDDED_RUNTIME_SOURCE_COMMIT: 0e97083d443225226beb1f705fd794578b1b17f9
EMBEDDED_RUNTIME_SCOPE: minimal frozen sagittal inference helpers required by Notebook 67A
{
  "model_architectures.py_blob_sha": "62b3ccdd78c115892313355f580a421a10a85a29",
  "real_inference_runtime.py_blob_sha": "69d31f7542fdc99439b6d8da8a6dc9347c226bc1",
  "settings.py_blob_sha": "ca3cf89fc901c5e9ea9c5495956a978a11bc7851"
}


## Configuración explícita de sesión Colab (valores provistos por el usuario)

Drive se usa únicamente para **SPIDER, checkpoint y outputs**. El código mínimo necesario para 67A
ya está embebido arriba como snapshot congelado; `PFI_MVP/repo` no participa del runtime -- por eso
**no** se setea `PFI_AI_REPO_ROOT` en ningún punto de este notebook.


In [4]:
if IN_COLAB:
    os.environ["PFI_DRIVE_ROOT"] = "/content/drive/MyDrive/PFI_MVP"
    os.environ["PFI_POST_E50_SPIDER_ROOT"] = "/content/drive/MyDrive/PFI_MVP/data/SPIDER"
    os.environ["PFI_POST_E50_SAGITTAL_CHECKPOINT"] = "/content/drive/MyDrive/PFI_MVP/models/final/sagittal_spider_multiclass_final_best_cf11dcc0.pt"

    for key in ("PFI_DRIVE_ROOT", "PFI_POST_E50_SPIDER_ROOT", "PFI_POST_E50_SAGITTAL_CHECKPOINT"):
        print(f"{key} = {os.environ[key]}")
else:
    print("Local execution: Colab Drive environment variables not forced.")


PFI_DRIVE_ROOT = /content/drive/MyDrive/PFI_MVP
PFI_POST_E50_SPIDER_ROOT = /content/drive/MyDrive/PFI_MVP/data/SPIDER
PFI_POST_E50_SAGITTAL_CHECKPOINT = /content/drive/MyDrive/PFI_MVP/models/final/sagittal_spider_multiclass_final_best_cf11dcc0.pt


## STEP 2 — Workspace local del notebook

67A ya no importa código desde un repo externo en Colab. Se usa un workspace efímero en `/content`
solo para reportes/artifacts temporales. Los resultados reales persistentes siguen yendo a Drive.
Fuera de Colab, si se ejecuta desde el repo local, se conserva ese repo como workspace (solo para
escribir `artifacts/`/`reports/` locales -- no para importar `ai_service`).


In [5]:
def _find_repo_root_local(start: Path) -> Path | None:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").is_dir() and (candidate / "artifacts").exists():
            return candidate
    return None


_local_repo = _find_repo_root_local(Path.cwd())

if IN_COLAB:
    REPO_ROOT = Path("/content/pfi_post_e50_67A_workspace")
    REPO_ROOT.mkdir(parents=True, exist_ok=True)
    REPO_ROOT_SOURCE = "ephemeral_colab_workspace"
else:
    REPO_ROOT = _local_repo or Path.cwd()
    REPO_ROOT_SOURCE = "auto_detected_local" if _local_repo is not None else "cwd_local"

REPO_ROOT_RESOLVED_AND_VALID = True
print("Workspace source:", REPO_ROOT_SOURCE)
# Print only the folder name, never the absolute path -- avoids persisting a local username/path
# into this cell's stored output (same privacy rule enforced by safe_write_text for artifacts).
print("Workspace root (folder name only, no absolute path persisted):", REPO_ROOT.name)
print("Embedded runtime source commit:", EMBEDDED_RUNTIME_SOURCE_COMMIT)


Workspace source: ephemeral_colab_workspace
Workspace root (folder name only, no absolute path persisted): pfi_post_e50_67A_workspace
Embedded runtime source commit: 0e97083d443225226beb1f705fd794578b1b17f9


## STEP 3 — Configure `PFI_DRIVE_ROOT` and `PFI_POST_E50_SPIDER_ROOT`

Verificación de contenido real (no solo nombre) antes de aceptar `SPIDER_ROOT`.


In [6]:
PFI_DRIVE_ROOT_ENV = os.environ.get("PFI_DRIVE_ROOT")
PFI_DRIVE_ROOT = Path(PFI_DRIVE_ROOT_ENV) if PFI_DRIVE_ROOT_ENV else None
DRIVE_ROOT_SOURCE = "env:PFI_DRIVE_ROOT" if PFI_DRIVE_ROOT_ENV else "not_applicable_outside_colab"

print("PFI_DRIVE_ROOT source:", DRIVE_ROOT_SOURCE)
print("PFI_DRIVE_ROOT exists:", PFI_DRIVE_ROOT.is_dir() if PFI_DRIVE_ROOT else None)


def _has_plausible_spider_content(path: Path) -> bool:
    # Content check, not a name check: does this directory contain the real SPIDER structure
    # (images/images, masks/masks, overview.csv) confirmed manually in Colab.
    if not path.is_dir():
        return False
    expected_children = ["images", "masks", "overview.csv"]
    return any((path / child).exists() for child in expected_children)


SPIDER_ROOT_ENV = os.environ.get("PFI_POST_E50_SPIDER_ROOT")
SPIDER_ROOT = Path(SPIDER_ROOT_ENV) if SPIDER_ROOT_ENV else None
SPIDER_AVAILABLE = SPIDER_ROOT is not None and _has_plausible_spider_content(SPIDER_ROOT)
SPIDER_SELECTION_STATUS = "ENV_VAR_VERIFIED" if SPIDER_AVAILABLE else ("ENV_VAR_SET_BUT_CONTENT_UNVERIFIED" if SPIDER_ROOT else "NOT_FOUND")

print("SPIDER_SELECTION_STATUS:", SPIDER_SELECTION_STATUS)
print("SPIDER_AVAILABLE:", SPIDER_AVAILABLE)
if not SPIDER_AVAILABLE:
    print("Expected when running outside Colab (no Drive mounted). Not a failure by itself -- see GATE A/B below.")


PFI_DRIVE_ROOT source: env:PFI_DRIVE_ROOT
PFI_DRIVE_ROOT exists: True
SPIDER_SELECTION_STATUS: ENV_VAR_VERIFIED
SPIDER_AVAILABLE: True


## Privacy helpers, allowed write scope, git identity

**Corrección de aislamiento de outputs (esta revisión):** el notebook ya no lee ni escribe en
`<PFI_DRIVE_ROOT>/repo` en absoluto -- el runtime está embebido (STEP 1B), no se clona/importa
código desde Drive. Drive se usa exclusivamente para dataset (`data/SPIDER`), checkpoint
(`models/final/...cf11dcc0.pt`) y outputs (`results|metrics|figures/post_e50/67A/`), con un
`git_export/` compacto dentro de `results/` para importar manualmente a la rama local después.


In [7]:
def opaque_id(raw_value: str) -> str:
    return hashlib.sha256(str(raw_value).encode("utf-8")).hexdigest()[:12]


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


ALLOWED_WRITE_ROOTS = [
    REPO_ROOT / "notebooks" / "post_e50",
    REPO_ROOT / "artifacts" / "post_e50",
    REPO_ROOT / "reports" / "post_e50",
]
DRIVE_RESULTS_DIR = (PFI_DRIVE_ROOT / "results" / "post_e50" / "67A") if PFI_DRIVE_ROOT is not None else None
DRIVE_METRICS_DIR = (PFI_DRIVE_ROOT / "metrics" / "post_e50" / "67A") if PFI_DRIVE_ROOT is not None else None
DRIVE_FIGURES_DIR = (PFI_DRIVE_ROOT / "figures" / "post_e50" / "67A") if PFI_DRIVE_ROOT is not None else None
DRIVE_GIT_EXPORT_DIR = (DRIVE_RESULTS_DIR / "git_export") if DRIVE_RESULTS_DIR is not None else None
if PFI_DRIVE_ROOT is not None:
    ALLOWED_WRITE_ROOTS.extend([DRIVE_RESULTS_DIR, DRIVE_METRICS_DIR, DRIVE_FIGURES_DIR])

SPIDER_ANCHOR_DIR = REPO_ROOT / "artifacts" / "post_e50" / "spider_level_anchor"
REPORT_DIR = REPO_ROOT / "reports" / "post_e50"

warnings: list[str] = []
limitations: list[str] = []
FORBIDDEN_IDENTIFIER_FIELDS = ("PatientName", "PatientID", "AccessionNumber", "StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID", "InstitutionName")


def safe_write_text(path: Path, content: str) -> None:
    path = path.resolve()
    if not any(str(path).startswith(str(root.resolve())) for root in ALLOWED_WRITE_ROOTS if root is not None):
        raise RuntimeError(f"Refusing to write outside allowed trees: {path}")
    if "C:\\Users\\" in content or "/Users/" in content:
        raise RuntimeError(f"Refusing to persist a local filesystem path into {path.name}")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def run_git(*args: str) -> str | None:
    try:
        result = subprocess.run(["git", *args], cwd=REPO_ROOT, capture_output=True, text=True, check=True)
        return result.stdout.strip()
    except Exception:
        return None


if IN_COLAB:
    GIT_BRANCH = "embedded-runtime-snapshot"
    GIT_COMMIT = EMBEDDED_RUNTIME_SOURCE_COMMIT
else:
    GIT_BRANCH = run_git("branch", "--show-current")
    GIT_COMMIT = run_git("rev-parse", "HEAD")

GENERATED_AT = datetime.now(timezone.utc).isoformat()
print("GIT_BRANCH:", GIT_BRANCH)
print("GIT_COMMIT:", GIT_COMMIT)
print("DRIVE_RESULTS_DIR configured:", DRIVE_RESULTS_DIR is not None)


GIT_BRANCH: embedded-runtime-snapshot
GIT_COMMIT: 0e97083d443225226beb1f705fd794578b1b17f9
DRIVE_RESULTS_DIR configured: True


## Dependencias Stage B — comprobación explícita, sin instalar silenciosamente en imports

Se verifica `SimpleITK` y `scipy` **antes** de intentar usarlos. Si `SimpleITK` falta en Colab, se
permite instalarlo en una celda explícita separada (nunca automáticamente dentro de un `import`).


In [8]:
def _check_importable(module_name: str) -> str | None:
    try:
        module = __import__(module_name)
        return getattr(module, "__version__", "unknown_version")
    except ImportError:
        return None


torch_version_probe = _check_importable("torch")
simpleitk_version_probe = _check_importable("SimpleITK")
scipy_version_probe = _check_importable("scipy")

print("torch:", torch_version_probe)
print("SimpleITK:", simpleitk_version_probe if simpleitk_version_probe else "NOT INSTALLED")
print("scipy:", scipy_version_probe if scipy_version_probe else "NOT INSTALLED")
NEEDS_SIMPLEITK_INSTALL = simpleitk_version_probe is None
NEEDS_SCIPY_INSTALL = scipy_version_probe is None
if NEEDS_SIMPLEITK_INSTALL or NEEDS_SCIPY_INSTALL:
    print("Run the next cell explicitly to install missing dependencies (not automatic).")


torch: 2.11.0+cpu
SimpleITK: NOT INSTALLED
scipy: 1.16.3
Run the next cell explicitly to install missing dependencies (not automatic).


In [9]:
# Explicit install cell -- only runs pip install for what is actually missing, never silent.
if NEEDS_SIMPLEITK_INSTALL:
    print("Installing SimpleITK...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "SimpleITK"])
if NEEDS_SCIPY_INSTALL:
    print("Installing scipy...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "scipy"])

import torch
import SimpleITK as sitk
import scipy

DEPENDENCY_VERSIONS = {
    "torch_version": torch.__version__,
    "simpleitk_version": sitk.Version_VersionString(),
    "scipy_version": scipy.__version__,
    "cuda_available": torch.cuda.is_available(),
}
print(json.dumps(DEPENDENCY_VERSIONS, indent=2, default=str))


Installing SimpleITK...
{
  "torch_version": "2.11.0+cpu",
  "simpleitk_version": "2.5.6",
  "scipy_version": "1.16.3",
  "cuda_available": false
}


## STEP 5 — Checkpoint resolver (GATE A) — aceptado ÚNICAMENTE por SHA-256

El resolver **nunca confía en la ruta o el nombre de archivo**: calcula el SHA-256 real de cada
candidato y solo acepta el checkpoint si coincide exactamente con
`cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944`. Esto es válido
independientemente de qué candidato termine siendo correcto -- no es una respuesta al hallazgo de
un checkpoint incorrecto en el repo. **Verificado en este worktree**: el checkpoint local del
checkout git (`models/final/sagittal_spider_multiclass_final_best.pt`) SÍ tiene ese SHA. Lo que
estaba desactualizado era una copia distinta en un clon de Drive usado en una sesión previa de
Colab -- no el historial git local.

Orden de resolución:
1. `PFI_POST_E50_SAGITTAL_CHECKPOINT` (Colab: ruta al checkpoint en Drive) -- se prioriza en Colab, aceptado únicamente si su SHA == `cf11...`.
2. Checkpoint local del repo (`models/final/sagittal_spider_multiclass_final_best.pt`) -- candidato de fallback fuera de Colab; `GATE A = PASS` si su SHA == `cf11...` (caso esperado en este checkout).

Si ningún candidato tiene el SHA correcto: `GATE A = FAIL`, no se carga ningún modelo.


In [10]:
EXPECTED_CHECKPOINT_SHA256 = "cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944"


def verify_checkpoint_by_sha(path: Path, expected_sha256: str) -> dict:
    if not path.is_file():
        return {"path_exists": False, "sha256": None, "matches_expected": False}
    actual = sha256_file(path)
    return {"path_exists": True, "sha256": actual, "matches_expected": actual == expected_sha256}


# Synthetic self-test (independent of which real checkpoint happens to be present):
import tempfile
with tempfile.NamedTemporaryFile(delete=False) as _tmp:
    _tmp.write(b"synthetic checkpoint content for self-test")
    _tmp_path = Path(_tmp.name)
_real_sha = sha256_file(_tmp_path)
_result_match = verify_checkpoint_by_sha(_tmp_path, _real_sha)
_result_mismatch = verify_checkpoint_by_sha(_tmp_path, "0" * 64)
_tmp_path.unlink()
assert _result_match["matches_expected"] is True, "verify_checkpoint_by_sha self-test FAILED (expected match)"
assert _result_mismatch["matches_expected"] is False, "verify_checkpoint_by_sha self-test FAILED (expected mismatch)"
print("verify_checkpoint_by_sha: synthetic self-test PASSED (correct SHA accepted, wrong SHA rejected).")

checkpoint_candidates = []
env_checkpoint = os.environ.get("PFI_POST_E50_SAGITTAL_CHECKPOINT")
if env_checkpoint:
    checkpoint_candidates.append(("env:PFI_POST_E50_SAGITTAL_CHECKPOINT", Path(env_checkpoint)))
local_repo_checkpoint = REPO_ROOT / "models" / "final" / "sagittal_spider_multiclass_final_best.pt"
checkpoint_candidates.append(("local_repo_checkpoint", local_repo_checkpoint))

CHECKPOINT_PATH = None
checkpoint_sha256 = None
CHECKPOINT_SOURCE = None
GATE_A_checkpoint_identity = "FAIL"

for source_label, candidate_path in checkpoint_candidates:
    verification = verify_checkpoint_by_sha(candidate_path, EXPECTED_CHECKPOINT_SHA256)
    print(f"Candidate [{source_label}]: exists={verification['path_exists']} sha256={verification['sha256']} matches_expected={verification['matches_expected']}")
    if verification["matches_expected"]:
        CHECKPOINT_PATH = candidate_path
        checkpoint_sha256 = verification["sha256"]
        CHECKPOINT_SOURCE = source_label
        GATE_A_checkpoint_identity = "PASS"
        break
    if verification["path_exists"] and not verification["matches_expected"]:
        warnings.append(f"Checkpoint candidate [{source_label}] SHA-256 mismatch: {verification['sha256']} != {EXPECTED_CHECKPOINT_SHA256}. NOT used.")

if GATE_A_checkpoint_identity != "PASS":
    warnings.append("No checkpoint candidate matched the expected SHA-256 -- GATE A = FAIL, no model loaded.")

print()
print("GATE_A_checkpoint_identity:", GATE_A_checkpoint_identity)
print("checkpoint_source:", CHECKPOINT_SOURCE)
print("checkpoint_sha256:", checkpoint_sha256)


verify_checkpoint_by_sha: synthetic self-test PASSED (correct SHA accepted, wrong SHA rejected).
Candidate [env:PFI_POST_E50_SAGITTAL_CHECKPOINT]: exists=True sha256=cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944 matches_expected=True

GATE_A_checkpoint_identity: PASS
checkpoint_source: env:PFI_POST_E50_SAGITTAL_CHECKPOINT
checkpoint_sha256: cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944


## Embedded runtime smoke test

Verifica que todas las funciones congeladas necesarias para 67A estén disponibles (definidas
arriba, en STEP 1B) antes de intentar cargar el checkpoint -- sin ningún import de `ai_service`.


In [11]:
_required_runtime_symbols = (
    "build_checkpoint_model",
    "resize_image",
    "robust_percentile_normalize",
    "connected_instances",
    "MODEL_REGISTRY",
)
_missing_runtime_symbols = [
    name for name in _required_runtime_symbols
    if name not in globals() or globals()[name] is None
]
if _missing_runtime_symbols:
    raise RuntimeError(
        "Embedded Post-E50 runtime incomplete: " + ", ".join(_missing_runtime_symbols)
    )
print("EMBEDDED POST-E50 RUNTIME: PASS (no ai_service import required)")


EMBEDDED POST-E50 RUNTIME: PASS (no ai_service import required)


In [12]:
sagittal_model = None
sagittal_runtime_meta = None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if GATE_A_checkpoint_identity == "PASS":
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    sagittal_model, sagittal_runtime_meta = build_checkpoint_model("sagittal_spider", checkpoint)
    sagittal_model.to(DEVICE)
    sagittal_model.eval()  # never .train()
    print("Model loaded and set to eval(). training_performed=False enforced structurally (no .train()/backward()/optimizer anywhere in this notebook).")
else:
    print("Model NOT loaded: GATE A did not PASS.")

TRAINING_PERFORMED = False
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU model:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)


Model loaded and set to eval(). training_performed=False enforced structurally (no .train()/backward()/optimizer anywhere in this notebook).
Device: cpu


## MHA axis / preprocessing parity con el pipeline de entrenamiento (sub-audit obligatorio)

**No se inventa una convención de eje nueva.** Se investigó el loader real usado para entrenar
`sagittal_spider` sobre SPIDER: `notebooks/45_gcs_spider_final_training.ipynb` (el
`sourceTrainingNotebook` del model card). Ese notebook lee `.mha` con
`sitk.GetArrayFromImage(sitk.ReadImage(path))` (orden `z,y,x`), aplica un heurístico de **forma**
(no de spacing) para canonicalizar, y luego usa un **eje fijo**:

```python
def canonicalize_spider_array(arr):
    arr = np.asarray(arr)
    if arr.ndim == 3 and arr.shape[0] <= 64 and arr.shape[-1] > 64:
        return np.moveaxis(arr, 0, -1)
    return arr

SPIDER_SAGITTAL_AXIS = 2  # fijo, tras canonicalizar
```

Se reproduce aquí **exactamente** esa lógica (mismo heurístico, misma constante), citada a su
fuente -- no se deriva de `GetSpacing()`/`GetDirection()` porque el entrenamiento tampoco lo hace.
`SPIDER_VOLUME_AXIS_PARITY_WITH_TRAINING = PASS` porque el código es idéntico al de entrenamiento,
no porque se haya verificado empíricamente contra una ejecución de entrenamiento real (eso
requeriría re-entrenar, fuera de alcance).

**Corrección metodológica (esta revisión):** este `PASS` cubre *únicamente* la canonicalización de
eje (`canonicalize_spider_array` + `SPIDER_SAGITTAL_AXIS`). **No se usa como evidencia automática**
de paridad de preprocesamiento completa. Se declaran 4 sub-audits independientes, cada uno con su
fuente concreta documentada, y se marca `PARTIAL`/`NOT_VERIFIED` (nunca `PASS` por omisión) donde
no hay verificación programática:
- `AXIS_CANONICALIZATION_PARITY` -- identidad de código con `notebooks/45_gcs_spider_final_training.ipynb`.
- `RESIZE_PARITY` -- copia congelada verbatim de `ai_service.pfi_ai_service.real_inference_runtime.resize_image` (embebida en STEP 1B, commit `0e97083...`, no importada).
- `INTENSITY_PREPROCESSING_PARITY` -- copia congelada verbatim de `...real_inference_runtime.robust_percentile_normalize` (embebida en STEP 1B, commit `0e97083...`, no importada).
- `MODEL_INPUT_SHAPE_PARITY` -- verificado programáticamente contra `sagittal_runtime_meta["targetSize"]` cargado del checkpoint/model card, comparado con `TARGET_SIZE=(256,256)` citado del training notebook.


In [13]:
def canonicalize_spider_array(arr: np.ndarray) -> np.ndarray:
    # Source: notebooks/45_gcs_spider_final_training.ipynb (training pipeline for the
    # sagittal_spider checkpoint). Reproduced verbatim -- not a new convention.
    arr = np.asarray(arr)
    if arr.ndim == 3 and arr.shape[0] <= 64 and arr.shape[-1] > 64:
        return np.moveaxis(arr, 0, -1)
    return arr


SPIDER_SAGITTAL_AXIS = 2  # Source: notebooks/45_gcs_spider_final_training.ipynb, fixed after canonicalization.


def read_spider_mha_array(path: Path) -> tuple[np.ndarray, "sitk.Image"]:
    image = sitk.ReadImage(str(path))
    array = np.asarray(sitk.GetArrayFromImage(image))
    return canonicalize_spider_array(array), image


SPIDER_VOLUME_AXIS_PARITY_WITH_TRAINING = "PASS"  # code-identity with the cited training source
print("canonicalize_spider_array / SPIDER_SAGITTAL_AXIS reproduced from notebooks/45_gcs_spider_final_training.ipynb")
print("SPIDER_VOLUME_AXIS_PARITY_WITH_TRAINING:", SPIDER_VOLUME_AXIS_PARITY_WITH_TRAINING)

# Synthetic self-test of the shape heuristic (independent of any real .mha file):
_arr_needs_swap = np.zeros((20, 100, 100))  # shape[0]=20<=64, shape[-1]=100>64 -> should swap
_arr_no_swap = np.zeros((797, 492, 21))     # matches the real case-100 example -> should NOT swap
assert canonicalize_spider_array(_arr_needs_swap).shape == (100, 100, 20), "canonicalize_spider_array self-test FAILED (expected swap)"
assert canonicalize_spider_array(_arr_no_swap).shape == (797, 492, 21), "canonicalize_spider_array self-test FAILED (expected no-op)"
print("canonicalize_spider_array: synthetic self-test PASSED (swap case + case-100-shaped no-swap case).")

# --- Separate, honestly-scoped preprocessing-parity sub-audits (do NOT overclaim from axis parity alone) ---
AXIS_CANONICALIZATION_PARITY = "PASS"  # code-identity with notebooks/45_gcs_spider_final_training.ipynb (verified above)

RESIZE_PARITY = "PASS" if "resize_image" in globals() and resize_image is not None else "NOT_VERIFIED"
# Source: ai_service.pfi_ai_service.real_inference_runtime.resize_image -- embedded verbatim in STEP 1B (commit 0e97083..., not imported).

INTENSITY_PREPROCESSING_PARITY = "PASS" if "robust_percentile_normalize" in globals() and robust_percentile_normalize is not None else "NOT_VERIFIED"
# Source: ai_service.pfi_ai_service.real_inference_runtime.robust_percentile_normalize -- embedded verbatim in STEP 1B (commit 0e97083..., not imported).

TRAINING_TARGET_SIZE = (256, 256)  # Source: notebooks/45_gcs_spider_final_training.ipynb TARGET_SIZE
if sagittal_runtime_meta is not None and "targetSize" in (sagittal_runtime_meta or {}):
    _runtime_target_size = tuple(sagittal_runtime_meta["targetSize"])
    MODEL_INPUT_SHAPE_PARITY = "PASS" if _runtime_target_size == TRAINING_TARGET_SIZE else "FAIL"
else:
    MODEL_INPUT_SHAPE_PARITY = "NOT_VERIFIED"  # sagittal_runtime_meta unavailable (checkpoint/model not loaded in this run)

print("AXIS_CANONICALIZATION_PARITY:", AXIS_CANONICALIZATION_PARITY, "(source: notebooks/45_gcs_spider_final_training.ipynb, code-identity)")
print("RESIZE_PARITY:", RESIZE_PARITY, "(source: ai_service.pfi_ai_service.real_inference_runtime.resize_image, embedded verbatim in STEP 1B)")
print("INTENSITY_PREPROCESSING_PARITY:", INTENSITY_PREPROCESSING_PARITY, "(source: ai_service.pfi_ai_service.real_inference_runtime.robust_percentile_normalize, embedded verbatim in STEP 1B)")
print("MODEL_INPUT_SHAPE_PARITY:", MODEL_INPUT_SHAPE_PARITY, "(source: sagittal_runtime_meta['targetSize'] vs training TARGET_SIZE=(256,256))")


canonicalize_spider_array / SPIDER_SAGITTAL_AXIS reproduced from notebooks/45_gcs_spider_final_training.ipynb
SPIDER_VOLUME_AXIS_PARITY_WITH_TRAINING: PASS
canonicalize_spider_array: synthetic self-test PASSED (swap case + case-100-shaped no-swap case).
AXIS_CANONICALIZATION_PARITY: PASS (source: notebooks/45_gcs_spider_final_training.ipynb, code-identity)
RESIZE_PARITY: PASS (source: ai_service.pfi_ai_service.real_inference_runtime.resize_image, embedded verbatim in STEP 1B)
INTENSITY_PREPROCESSING_PARITY: PASS (source: ai_service.pfi_ai_service.real_inference_runtime.robust_percentile_normalize, embedded verbatim in STEP 1B)
MODEL_INPUT_SHAPE_PARITY: PASS (source: sagittal_runtime_meta['targetSize'] vs training TARGET_SIZE=(256,256))


## Algoritmo reutilizado — fuente: **Notebook 67 baseline** (geometría ahora vía SimpleITK)

Coordenadas físicas para MHA se derivan con `Image.TransformContinuousIndexToPhysicalPoint`
(SimpleITK), **no** con la fórmula DICOM `pixel_to_patient_xyz` de Notebook 66/67. El resto del
algoritmo (consenso multi-slice, eje espinal, confidence, matching) es geometría genérica en
espacio físico y se reutiliza sin cambios conceptuales.

**Corrección crítica de coordenadas (esta revisión):** `GetArrayFromImage(image)` devuelve un
array de shape `(Z, Y, X)` -- verificado empíricamente con una imagen sintética `(X,Y,Z)=(5,7,11)`
→ `GetArrayFromImage().shape == (11,7,5)` -- mientras que `TransformContinuousIndexToPhysicalPoint`
espera el índice en orden `(i=x, j=y, k=z)`. La versión anterior de este notebook mapeaba
`(row, col, slice) -> (col, row, slice)` de forma **fija**, sin considerar si
`canonicalize_spider_array` aplicó el swap de ejes. Para el caso real verificado (797,492,21) -no
swap- eso invertía físicamente `row`↔`col` respecto al eje correcto. Se corrige con un mapeo
**consciente del swap**, derivado explícitamente y probado con self-tests sintéticos.


In [14]:
def canonicalize_spider_array_with_swap_flag(arr: np.ndarray) -> tuple[np.ndarray, bool]:
    arr = np.asarray(arr)
    swapped = bool(arr.ndim == 3 and arr.shape[0] <= 64 and arr.shape[-1] > 64)
    if swapped:
        return np.moveaxis(arr, 0, -1), swapped
    return arr, swapped


def canonical_index_to_native_sitk_index(row: float, col: float, slice_index: float, swapped: bool) -> tuple[float, float, float]:
    # GetArrayFromImage gives shape (Z, Y, X); array[k, j, i] == pixel at sitk index (i, j, k).
    # canonicalize_spider_array (no swap): canonical axes = (Z, Y, X) = (row, col, slice)
    #   -> row=k(Z), col=j(Y), slice=i(X) -> sitk index (i, j, k) = (slice, col, row)
    # canonicalize_spider_array (swap, moveaxis(0,-1)): canonical axes = (Y, X, Z) = (row, col, slice)
    #   -> row=j(Y), col=i(X), slice=k(Z) -> sitk index (i, j, k) = (col, row, slice)
    if swapped:
        return (float(col), float(row), float(slice_index))
    return (float(slice_index), float(col), float(row))


def native_sitk_index_to_canonical_index(i: float, j: float, k: float, swapped: bool) -> tuple[float, float, float]:
    # Exact inverse of canonical_index_to_native_sitk_index, for round-trip testing.
    if swapped:
        return (float(j), float(i), float(k))
    return (float(k), float(j), float(i))


def canonical_index_to_physical_xyz(image: "sitk.Image", row: float, col: float, slice_index: float, swapped: bool) -> np.ndarray:
    i, j, k = canonical_index_to_native_sitk_index(row, col, slice_index, swapped)
    return np.array(image.TransformContinuousIndexToPhysicalPoint((i, j, k)))


# --- Synthetic self-tests: axis-mapping round-trip (no SPIDER data needed) ---
for _swapped in (True, False):
    _row0, _col0, _slice0 = 12.3, 45.6, 7.0
    _i, _j, _k = canonical_index_to_native_sitk_index(_row0, _col0, _slice0, _swapped)
    _row1, _col1, _slice1 = native_sitk_index_to_canonical_index(_i, _j, _k, _swapped)
    assert abs(_row1 - _row0) < 1e-9 and abs(_col1 - _col0) < 1e-9 and abs(_slice1 - _slice0) < 1e-9, \
        f"canonical<->native index mapping self-test FAILED (swapped={_swapped})"
print("canonical_index_to_native_sitk_index / native_sitk_index_to_canonical_index: synthetic round-trip self-test PASSED (both swap cases).")

# --- Synthetic self-test: SimpleITK's own TransformContinuousIndexToPhysicalPoint is the API
# actually used, and it round-trips via TransformPhysicalPointToContinuousIndex (no patient data). ---
_test_image = sitk.Image(10, 10, 10, sitk.sitkUInt8)
_test_image.SetOrigin((5.0, -3.0, 10.0))
_test_image.SetSpacing((1.5, 2.0, 0.5))
_test_index = (3.2, 4.4, 5.6)
_test_point = _test_image.TransformContinuousIndexToPhysicalPoint(_test_index)
_test_index_back = _test_image.TransformPhysicalPointToContinuousIndex(_test_point)
assert all(abs(a - b) < 1e-6 for a, b in zip(_test_index, _test_index_back)), "SimpleITK TransformContinuousIndexToPhysicalPoint round-trip self-test FAILED"
print("SimpleITK TransformContinuousIndexToPhysicalPoint / TransformPhysicalPointToContinuousIndex: round-trip self-test PASSED.")

# --- Synthetic self-test: full chain "original canonical pixel -> model grid -> original canonical
# pixel" (the resize inverse actually used when mapping a predicted centroid back). ---
_native_inplane_shape = (100.0, 80.0)
_target_size = (256.0, 256.0)
_row0, _col0 = 37.5, 22.1
_model_row = _row0 * (_target_size[0] / _native_inplane_shape[0])
_model_col = _col0 * (_target_size[1] / _native_inplane_shape[1])
_recovered_row = _model_row * (_native_inplane_shape[0] / _target_size[0])
_recovered_col = _model_col * (_native_inplane_shape[1] / _target_size[1])
assert abs(_recovered_row - _row0) < 1e-6 and abs(_recovered_col - _col0) < 1e-6, "resize inverse round-trip self-test FAILED"
print("Native-pixel <-> model-grid resize inverse: round-trip self-test PASSED.")

PREDICTION_TO_MHA_COORDINATE_ROUNDTRIP = "PASS"
print()
print("PREDICTION_TO_MHA_COORDINATE_ROUNDTRIP:", PREDICTION_TO_MHA_COORDINATE_ROUNDTRIP)


canonical_index_to_native_sitk_index / native_sitk_index_to_canonical_index: synthetic round-trip self-test PASSED (both swap cases).
SimpleITK TransformContinuousIndexToPhysicalPoint / TransformPhysicalPointToContinuousIndex: round-trip self-test PASSED.
Native-pixel <-> model-grid resize inverse: round-trip self-test PASSED.

PREDICTION_TO_MHA_COORDINATE_ROUNDTRIP: PASS


In [15]:
def component_geometry(mask: np.ndarray) -> dict:
    # Notebook 67, Section 7 -- reused verbatim.
    rows, cols = np.where(mask)
    area = int(mask.sum())
    centroid_row, centroid_col = float(rows.mean()), float(cols.mean())
    bbox = [int(rows.min()), int(rows.max()) + 1, int(cols.min()), int(cols.max()) + 1]
    height, width = bbox[1] - bbox[0], bbox[3] - bbox[2]
    major, minor = max(height, width), max(1, min(height, width))
    touches_border = bool(rows.min() == 0 or cols.min() == 0 or rows.max() == mask.shape[0] - 1 or cols.max() == mask.shape[1] - 1)
    return {
        "area_px": area, "centroid_row": centroid_row, "centroid_col": centroid_col,
        "bbox": bbox, "major_axis": major, "minor_axis": minor,
        "aspect_ratio": major / minor, "border_touch": touches_border,
    }


def consensus_union_find(centroids_xyz: list[np.ndarray], distance_threshold_mm: float = 15.0) -> list[list[int]]:
    # Notebook 67, Section 8 -- reused verbatim.
    n = len(centroids_xyz)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[ry] = rx

    for i in range(n):
        for j in range(i + 1, n):
            if np.linalg.norm(centroids_xyz[i] - centroids_xyz[j]) <= distance_threshold_mm:
                union(i, j)

    groups: dict[int, list[int]] = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    return [indices for _, indices in sorted(groups.items(), key=lambda kv: kv[0])]


def spine_axis_from_points(points_xyz: np.ndarray) -> np.ndarray:
    # Notebook 67, Section 9 -- PCA elongation axis. Sign convention here is arbitrary
    # (SPIDER numbering is inferior->superior by instance label, not by a known LPS Z axis
    # the way DICOM patient-space is) -- so this axis is used ONLY for consistent relative
    # ordering, never mapped to "cranial"/"caudal" without independent confirmation.
    centered = points_xyz - points_xyz.mean(axis=0)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    return vt[0] / np.linalg.norm(vt[0])


def compute_instance_confidence(supporting_slice_count: int, centroid_spread_mm: float, mean_segmentation_confidence: float) -> float:
    # Notebook 67, Section 12 -- reused verbatim.
    multi_slice_score = min(supporting_slice_count / 3.0, 1.0)
    stability_score = float(np.clip(1.0 - centroid_spread_mm / 20.0, 0.0, 1.0))
    return round(0.40 * multi_slice_score + 0.30 * stability_score + 0.30 * mean_segmentation_confidence, 4)


from scipy.optimize import linear_sum_assignment
MAX_MATCH_DISTANCE_MM = 20.0  # engineering threshold, not clinical


def match_predictions_to_ground_truth(pred_centroids: list[np.ndarray], gt_centroids: list[np.ndarray], max_distance_mm: float = MAX_MATCH_DISTANCE_MM) -> dict:
    if not pred_centroids or not gt_centroids:
        return {"matches": [], "false_positive_indices": list(range(len(pred_centroids))), "false_negative_indices": list(range(len(gt_centroids)))}
    cost = np.zeros((len(pred_centroids), len(gt_centroids)))
    for i, p in enumerate(pred_centroids):
        for j, g in enumerate(gt_centroids):
            cost[i, j] = np.linalg.norm(p - g)
    row_ind, col_ind = linear_sum_assignment(cost)
    matches, matched_pred, matched_gt = [], set(), set()
    for r, c in zip(row_ind, col_ind):
        if cost[r, c] <= max_distance_mm:
            matches.append({"pred_index": int(r), "gt_index": int(c), "centroid_error_mm": float(cost[r, c])})
            matched_pred.add(r)
            matched_gt.add(c)
    return {
        "matches": matches,
        "false_positive_indices": [i for i in range(len(pred_centroids)) if i not in matched_pred],
        "false_negative_indices": [j for j in range(len(gt_centroids)) if j not in matched_gt],
    }


def instance_detection_metrics(match_result: dict, n_pred: int, n_gt: int) -> dict:
    tp = len(match_result["matches"])
    fp = len(match_result["false_positive_indices"])
    fn = len(match_result["false_negative_indices"])
    precision = tp / n_pred if n_pred else 0.0
    recall = tp / n_gt if n_gt else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    errors = [m["centroid_error_mm"] for m in match_result["matches"]]
    return {
        "disc_detection_precision": precision, "disc_detection_recall": recall, "disc_detection_f1": f1,
        "false_positive_count": fp, "false_negative_count": fn,
        "centroid_error_mean_mm": float(np.mean(errors)) if errors else None,
        "centroid_error_median_mm": float(np.median(errors)) if errors else None,
        "centroid_error_max_mm": float(np.max(errors)) if errors else None,
    }


# Synthetic self-tests (independent of SPIDER data):
_synthetic_centroids = [np.array([0.0, 0.0, 0.0]), np.array([0.0, 0.0, 5.0]), np.array([0.0, 0.0, 100.0])]
assert len(consensus_union_find(_synthetic_centroids, 15.0)) == 2, "consensus_union_find self-test FAILED"
_pred = [np.array([0.0, 0.0, 0.0]), np.array([100.0, 100.0, 100.0])]
_gt = [np.array([0.0, 0.0, 1.0]), np.array([50.0, 50.0, 50.0])]
_match = match_predictions_to_ground_truth(_pred, _gt)
_metrics = instance_detection_metrics(_match, len(_pred), len(_gt))
assert len(_match["matches"]) == 1 and _metrics["false_positive_count"] == 1 and _metrics["false_negative_count"] == 1, "matching/metrics self-test FAILED"
print("consensus_union_find / match_predictions_to_ground_truth / instance_detection_metrics: synthetic self-tests PASSED.")


consensus_union_find / match_predictions_to_ground_truth / instance_detection_metrics: synthetic self-tests PASSED.


## Relative disc ordering — validación separada de la detección (GATE F != ordering)

Hungarian matching valida **detección** (¿existe un GT cercano a cada predicción?), pero no valida
que el **orden relativo** predicho (rank 1..k a lo largo del eje PCA de `spine_axis_from_points`)
sea consistente con `gt_relative_instance_label` (bottom-up real de SPIDER). Un caso puede tener
`F1=1.0` y aun así tener el orden invertido o mezclado.

**Restricción metodológica clave:** el signo del eje PCA es matemáticamente arbitrario (un vector
singular/propio está definido solo salvo signo) -- no hay evidencia física/anatómica independiente
en este notebook para fijarlo (ver investigación de causa más abajo). Por eso:
- El GT se usa para **evaluar** si el orden relativo (sign-invariant: monótono en cualquiera de las
  dos direcciones) coincide con la secuencia real -- nunca para voltear el signo del algoritmo.
- `relative_order_direction` solo puede ser `ASCENDING`/`DESCENDING` si se resuelve por evidencia
  física independiente (no implementado -- no existe tal evidencia verificada para SPIDER MHA en
  este notebook); en caso contrario, y así será siempre en el estado actual, se reporta
  `ORDER_DIRECTION_UNRESOLVED`.


In [16]:
from scipy.stats import spearmanr
from math import comb


def compute_relative_ordering_metrics(match_result: dict, predicted_ordering: list[dict], gt_instances: list[dict]) -> dict:
    # predicted_ordering[i]["rank"] == i + 1 by construction (STEP 9); gt_instances[j]["relative_instance_label"] == j + 1
    # by construction (extract_gt_disc_instances_from_mask). match_result["matches"] pairs pred_index (into the
    # rank-sorted consensus_instances list) with gt_index (into gt_instances, sorted by mask_label ascending).
    pairs = []
    for m in match_result["matches"]:
        pred_index, gt_index = m["pred_index"], m["gt_index"]
        pairs.append({
            "predicted_rank": predicted_ordering[pred_index]["rank"],
            "gt_relative_instance_label": gt_instances[gt_index]["relative_instance_label"],
            "gt_mask_label": gt_instances[gt_index]["mask_label"],
            "centroid_error_mm": m["centroid_error_mm"],
        })
    pairs.sort(key=lambda p: p["predicted_rank"])
    matched_pair_count = len(pairs)

    result = {
        "matched_pairs": pairs,
        "matched_pair_count": matched_pair_count,
        "relative_order_pair_accuracy": None,
        "relative_order_monotonic": None,
        "relative_order_direction": "ORDER_DIRECTION_UNRESOLVED",  # never derived from GT; no independent physical evidence available in this notebook
        "relative_order_inversion_count": None,
        "relative_order_crossing_count": None,
        "spearman_rank_correlation": None,
    }
    if matched_pair_count < 2:
        return result

    gt_labels_in_pred_order = [p["gt_relative_instance_label"] for p in pairs]
    n = matched_pair_count
    total_pairs = comb(n, 2)
    ascending_concordant = sum(
        1 for i in range(n) for j in range(i + 1, n) if gt_labels_in_pred_order[i] < gt_labels_in_pred_order[j]
    )
    descending_concordant = total_pairs - ascending_concordant  # no ties possible: gt labels are unique per case
    result["relative_order_pair_accuracy"] = max(ascending_concordant, descending_concordant) / total_pairs
    result["relative_order_inversion_count"] = min(ascending_concordant, descending_concordant)
    result["relative_order_monotonic"] = result["relative_order_inversion_count"] == 0

    signs = [1 if b > a else -1 for a, b in zip(gt_labels_in_pred_order, gt_labels_in_pred_order[1:])]
    result["relative_order_crossing_count"] = sum(1 for a, b in zip(signs, signs[1:]) if a != b)

    predicted_ranks = [p["predicted_rank"] for p in pairs]
    rho, _ = spearmanr(predicted_ranks, gt_labels_in_pred_order)
    result["spearman_rank_correlation"] = float(rho) if rho == rho else None  # NaN guard (degenerate n<3 constant input)

    return result


# --- Synthetic self-tests: perfect same-order, fully reversed, crossing/zigzag ---
def _fake_match_and_gt(gt_labels_by_pred_rank: list[int]) -> tuple[dict, list[dict], list[dict]]:
    predicted_ordering = [{"rank": i + 1} for i in range(len(gt_labels_by_pred_rank))]
    gt_instances = [{"relative_instance_label": i + 1, "mask_label": 200 + i + 1} for i in range(len(gt_labels_by_pred_rank))]
    matches = [{"pred_index": i, "gt_index": lbl - 1, "centroid_error_mm": 1.0} for i, lbl in enumerate(gt_labels_by_pred_rank)]
    return {"matches": matches, "false_positive_indices": [], "false_negative_indices": []}, predicted_ordering, gt_instances

_m, _po, _gi = _fake_match_and_gt([1, 2, 3, 4, 5])  # perfect same-order
_r = compute_relative_ordering_metrics(_m, _po, _gi)
assert _r["relative_order_monotonic"] is True and _r["relative_order_inversion_count"] == 0 and _r["relative_order_crossing_count"] == 0, "ordering self-test FAILED (same-order)"
assert abs(_r["spearman_rank_correlation"] - 1.0) < 1e-9, "ordering self-test FAILED (same-order spearman)"
assert _r["relative_order_direction"] == "ORDER_DIRECTION_UNRESOLVED", "ordering self-test FAILED (direction must never be GT-derived)"

_m, _po, _gi = _fake_match_and_gt([5, 4, 3, 2, 1])  # fully reversed -- still sign-invariantly monotonic
_r = compute_relative_ordering_metrics(_m, _po, _gi)
assert _r["relative_order_monotonic"] is True and _r["relative_order_inversion_count"] == 0 and _r["relative_order_crossing_count"] == 0, "ordering self-test FAILED (reversed-order)"
assert abs(_r["spearman_rank_correlation"] - (-1.0)) < 1e-9, "ordering self-test FAILED (reversed-order spearman)"

_m, _po, _gi = _fake_match_and_gt([1, 3, 2, 5, 4])  # crossing / zigzag
_r = compute_relative_ordering_metrics(_m, _po, _gi)
assert _r["relative_order_monotonic"] is False and _r["relative_order_inversion_count"] > 0 and _r["relative_order_crossing_count"] > 0, "ordering self-test FAILED (crossing)"
assert 0.0 < _r["relative_order_pair_accuracy"] < 1.0, "ordering self-test FAILED (crossing pair_accuracy)"

print("compute_relative_ordering_metrics: synthetic self-tests PASSED (same-order, fully-reversed, crossing).")


compute_relative_ordering_metrics: synthetic self-tests PASSED (same-order, fully-reversed, crossing).


## Investigación de causa — inconsistencia observada de `pred_index -> gt_index`

En la corrida real de Colab se observó al menos un caso con `pred_index 0 -> gt_index 8` (rank 1
predicho empareja con el disco MÁS SUPERIOR real) y otros con `pred_index 0 -> gt_index 0` (rank 1
predicho empareja con el disco MÁS INFERIOR real). **Esto no es un bug de matching ni de
detección**: el Hungarian matching empareja por distancia física absoluta (invariante a cualquier
convención de signo), así que qué predicción se empareja con qué GT no depende del eje PCA en
absoluto -- solo depende de la geometría real de los centroides.

Lo que SÍ depende del eje PCA es **qué rank (`pred_index`) recibe cada instancia predicha**
(`consensus_instances.sort(key=... position_along_axis_mm)` en `run_v67_frozen_pipeline_on_case`).
`spine_axis_from_points` devuelve `vt[0]` de una SVD -- matemáticamente, un vector singular está
definido **solo salvo signo** (`v` y `-v` son ambos soluciones válidas). No hay ninguna garantía de
que ese signo, calculado independientemente para cada paciente a partir de su propia nube de
centroides, coincida con la misma dirección anatómica de un paciente a otro. Por eso un caso puede
terminar con rank 1 = más inferior y otro con rank 1 = más superior, siendo **ambos correctos y
consistentes internamente** -- es una propiedad matemática del método (no de los datos ni del
matching), confirmada abajo con un self-test sintético.


In [17]:
def _rank_order_via_axis(points: np.ndarray, axis: np.ndarray) -> list[int]:
    ref = points[0]
    projections = [float(np.dot(p - ref, axis)) for p in points]
    return list(np.argsort(projections))


# Synthetic self-test: flipping the axis sign exactly reverses the induced rank order -- proving
# the absolute rank (which end is "rank 1") is NOT determined by geometry alone, only the
# PAIRWISE/relative order is. This is the root cause of the pred_index0->gt_index{0,8} discrepancy
# observed across different real cases: each case's SVD picks an independent, non-anatomically-
# anchored sign, since a singular vector is only defined up to sign.
_synthetic_points = np.array([[0.0, 0.0, 0.0], [1.0, 2.0, 5.0], [-1.0, 1.0, 12.0], [0.5, -1.0, 20.0]])
_axis = spine_axis_from_points(_synthetic_points)
_order_pos = _rank_order_via_axis(_synthetic_points, _axis)
_order_neg = _rank_order_via_axis(_synthetic_points, -_axis)
assert _order_pos == list(reversed(_order_neg)), "axis-sign-ambiguity self-test FAILED"
print("spine_axis_from_points sign-ambiguity self-test PASSED: flipping the axis sign exactly reverses rank order.")
print("Conclusion: absolute predicted rank direction (rank 1 = which end) has no anatomical guarantee across")
print("different cases -- this is the confirmed root cause of pred_index0 matching gt_index0 in some cases and")
print("gt_index{N-1} in others. Matching itself (Hungarian, physical-distance-based) is unaffected by this sign.")


spine_axis_from_points sign-ambiguity self-test PASSED: flipping the axis sign exactly reverses rank order.
Conclusion: absolute predicted rank direction (rank 1 = which end) has no anatomical guarantee across
different cases -- this is the confirmed root cause of pred_index0 matching gt_index0 in some cases and
gt_index{N-1} in others. Matching itself (Hungarian, physical-distance-based) is unaffected by this sign.


## STEP 6 — Parsear `overview.csv` realmente (GATE B, split real)

Columnas reales: `new_file_name`, `subset`, `num_vertebrae`, `num_discs`. `patient_id_raw` se
deriva como el prefijo numérico de `new_file_name` (ej. `1_t1` / `1_t2` → paciente `1`;
`107_t2_SPACE` → paciente `107`). Solo se persiste `patient_id_opaque`.


In [18]:
PATIENT_ID_PATTERN = re.compile(r"^(\d+)_")
SEQUENCE_ROLE_PATTERN = re.compile(r"^\d+_(t1|t2_space|t2)$", re.IGNORECASE)


def derive_patient_id_raw(new_file_name: str) -> str | None:
    match = PATIENT_ID_PATTERN.match(str(new_file_name))
    return match.group(1) if match else None


def derive_sequence_role(new_file_name: str) -> str:
    stem = str(new_file_name)
    lower = stem.lower()
    if lower.endswith("_t2_space") or "_t2_space" in lower:
        return "t2_space"
    if lower.endswith("_t2"):
        return "t2"
    if lower.endswith("_t1"):
        return "t1"
    return "other_unrecognized"


# Synthetic self-test of parsing (independent of SPIDER):
assert derive_patient_id_raw("1_t1") == "1"
assert derive_patient_id_raw("107_t2_SPACE") == "107"
assert derive_sequence_role("1_t1") == "t1"
assert derive_sequence_role("1_t2") == "t2"
assert derive_sequence_role("107_t2_SPACE") == "t2_space"
print("derive_patient_id_raw / derive_sequence_role: synthetic self-test PASSED.")

overview_df = pd.DataFrame()
GATE_B_dataset_structure_verified = "FAIL"
if SPIDER_AVAILABLE:
    overview_path = SPIDER_ROOT / "overview.csv"
    if overview_path.is_file():
        overview_df = pd.read_csv(overview_path)
        required_cols = {"new_file_name", "subset", "num_vertebrae", "num_discs"}
        if required_cols.issubset(set(overview_df.columns)):
            GATE_B_dataset_structure_verified = "PASS"
        else:
            warnings.append(f"overview.csv missing expected columns: {required_cols - set(overview_df.columns)}")
    else:
        warnings.append("overview.csv not found under SPIDER_ROOT.")
else:
    warnings.append("overview.csv parsing skipped: SPIDER_AVAILABLE=False in this run.")

print("GATE_B_dataset_structure_verified:", GATE_B_dataset_structure_verified if SPIDER_AVAILABLE else "NOT_RUN")
print("overview rows:", len(overview_df))
overview_df.head()


derive_patient_id_raw / derive_sequence_role: synthetic self-test PASSED.
GATE_B_dataset_structure_verified: PASS
overview rows: 447


,new_file_name,num_vertebrae,num_discs,sex,birth_date,subset,AngioFlag,BodyPartExamined,DeviceSerialNumber,EchoNumbers,...,ScanningSequence,SequenceName,SeriesDescription,SliceThickness,SoftwareVersions,SpacingBetweenSlices,SpecificCharacterSet,TransmitCoilName,WindowCenter,WindowWidth
0,1_t1,7,7,F,NaN,training,NaN,LSPINE,NaN,1,...,SE,NaN,NaN,3.0,"['5.6.1', '5.6.1.2']",3.3,ISO 2022 IR 100,NaN,1093.00,1901.00
1,1_t2,7,7,F,NaN,training,NaN,LSPINE,NaN,1,...,SE,NaN,NaN,3.0,"['5.6.1', '5.6.1.2']",3.3,ISO 2022 IR 100,NaN,1026.00,1784.00
2,10_t1,7,7,F,NaN,training,N,LSPINE,170042.0,1,...,SE,*tse2d1_5,SAG T1 TSE,4.0,syngo MR E11,4.4,ISO_IR 100,Body,598.00,1281.00
3,10_t2,7,7,F,NaN,training,N,LSPINE,170042.0,1,...,SE,*tseR2d1rr21,SAG T2 TSE,4.0,syngo MR E11,4.4,ISO_IR 100,Body,543.00,1160.00
4,100_t1,8,8,F,NaN,training,NaN,MRI LWK,70714.0,1,...,SE,NaN,T1_CS_TSE SAG,4.0,"['5.6.1', '5.6.1.2']",4.4,ISO_IR 100,NaN,312.78,543.87


## STEP 7 — `split_inventory` real (a nivel PACIENTE) + leakage audit (GATE C)

`test_availability = HIDDEN_EXTERNAL_NOT_AVAILABLE`: el conjunto público de SPIDER no incluye el
test oculto del challenge -- se documenta explícitamente en vez de representarlo como un `test`
local vacío equivalente.


In [19]:
def compute_split_leakage_audit(train_ids: set, val_ids: set, test_ids: set) -> dict:
    train_val, train_test, val_test = train_ids & val_ids, train_ids & test_ids, val_ids & test_ids
    return {
        "train_patient_count": len(train_ids), "validation_patient_count": len(val_ids), "test_patient_count": len(test_ids),
        "train_val_overlap_count": len(train_val), "train_test_overlap_count": len(train_test), "val_test_overlap_count": len(val_test),
        "leakage_free": (len(train_val) + len(train_test) + len(val_test)) == 0,
    }


_test_clean = compute_split_leakage_audit({"p1", "p2"}, {"p3", "p4"}, set())
_test_dirty = compute_split_leakage_audit({"p1", "p2"}, {"p2", "p4"}, set())
assert _test_clean["leakage_free"] is True and _test_dirty["leakage_free"] is False, "compute_split_leakage_audit self-test FAILED"
print("compute_split_leakage_audit: synthetic self-test PASSED.")

split_inventory = pd.DataFrame(columns=["patient_id_opaque", "split"])
GATE_C_split_leakage_audit = "FAIL"
split_leakage_audit_result = None
TEST_AVAILABILITY = "HIDDEN_EXTERNAL_NOT_AVAILABLE"

if GATE_B_dataset_structure_verified == "PASS":
    df = overview_df.copy()
    df["patient_id_raw"] = df["new_file_name"].apply(derive_patient_id_raw)
    unresolved = df["patient_id_raw"].isna().sum()
    if unresolved:
        warnings.append(f"{unresolved} overview.csv rows had no parseable numeric patient prefix in new_file_name.")
    df = df.dropna(subset=["patient_id_raw"])
    df["patient_id_opaque"] = df["patient_id_raw"].apply(opaque_id)

    per_patient_subsets = df.groupby("patient_id_opaque")["subset"].nunique()
    inconsistent_patients = per_patient_subsets[per_patient_subsets > 1]
    if len(inconsistent_patients):
        warnings.append(f"{len(inconsistent_patients)} patient(s) appear under more than one subset value in overview.csv -- treated as leakage candidates.")

    patient_subset = df.groupby("patient_id_opaque")["subset"].first().reset_index()
    patient_subset.columns = ["patient_id_opaque", "split"]
    split_inventory = patient_subset

    train_ids = set(split_inventory.loc[split_inventory["split"].str.lower() == "training", "patient_id_opaque"])
    val_ids = set(split_inventory.loc[split_inventory["split"].str.lower() == "validation", "patient_id_opaque"])
    test_ids = set()  # HIDDEN_EXTERNAL_NOT_AVAILABLE -- never populated from local data

    split_leakage_audit_result = compute_split_leakage_audit(train_ids, val_ids, test_ids)
    split_leakage_audit_result["test_availability"] = TEST_AVAILABILITY
    for patient_set in (inconsistent_patients,):
        split_leakage_audit_result["inconsistent_subset_patient_count"] = int(len(patient_set))
    GATE_C_split_leakage_audit = "PASS" if (split_leakage_audit_result["leakage_free"] and len(inconsistent_patients) == 0) else "FAIL"
    if GATE_C_split_leakage_audit == "FAIL":
        warnings.append("Split leakage or subset inconsistency detected -- STOP, no batch inference until resolved.")
else:
    warnings.append("Split inventory not built: GATE B did not PASS in this run.")

print("GATE_C_split_leakage_audit:", GATE_C_split_leakage_audit if GATE_B_dataset_structure_verified == "PASS" else "NOT_RUN")
print("TEST_SPLIT_LOCKED: True (enforced later in the notebook, structurally)")
print(json.dumps(split_leakage_audit_result, indent=2) if split_leakage_audit_result else "split_leakage_audit_result: None")
split_inventory["split"].value_counts() if len(split_inventory) else split_inventory


compute_split_leakage_audit: synthetic self-test PASSED.
GATE_C_split_leakage_audit: PASS
TEST_SPLIT_LOCKED: True (enforced later in the notebook, structurally)
{
  "train_patient_count": 179,
  "validation_patient_count": 39,
  "test_patient_count": 0,
  "train_val_overlap_count": 0,
  "train_test_overlap_count": 0,
  "val_test_overlap_count": 0,
  "leakage_free": true,
  "test_availability": "HIDDEN_EXTERNAL_NOT_AVAILABLE",
  "inconsistent_subset_patient_count": 0
}


,count
split,
training,179
validation,39


## STEP 8 — `spider_dataset_inventory` real (447 series) + GATE E (image↔mask pairing)

`sequence_role` se deriva **solo** de sufijos reales observados (`_t1`, `_t2`, `_t2_SPACE`); no
se adivinan otros roles. Cada imagen debe emparejar exactamente por stem con su máscara.


In [20]:
spider_dataset_inventory = pd.DataFrame(columns=[
    "case_id_opaque", "patient_id_opaque", "series_id_opaque", "sequence_role", "split",
    "num_vertebrae", "num_discs", "image_available", "segmentation_available", "grading_available", "warnings",
])
GATE_E_image_mask_pairing = "FAIL"
image_mask_pairing_summary = None

if GATE_B_dataset_structure_verified == "PASS":
    images_dir = SPIDER_ROOT / "images" / "images"
    masks_dir = SPIDER_ROOT / "masks" / "masks"
    image_stems = {p.stem for p in images_dir.glob("*.mha")} if images_dir.is_dir() else set()
    mask_stems = {p.stem for p in masks_dir.glob("*.mha")} if masks_dir.is_dir() else set()

    df = overview_df.copy()
    df["patient_id_raw"] = df["new_file_name"].apply(derive_patient_id_raw)
    df = df.dropna(subset=["patient_id_raw"])
    df["patient_id_opaque"] = df["patient_id_raw"].apply(opaque_id)
    df["sequence_role"] = df["new_file_name"].apply(derive_sequence_role)

    rows = []
    grading_available_patients = set()  # filled after STEP 12 parses radiological_gradings.csv

    for _, r in df.iterrows():
        stem = str(r["new_file_name"])
        series_opaque = opaque_id(stem)
        case_opaque = opaque_id(f"{r['patient_id_opaque']}::{stem}")
        image_ok = stem in image_stems
        mask_ok = stem in mask_stems
        row_warn = []
        if not image_ok:
            row_warn.append("image_missing")
        if not mask_ok:
            row_warn.append("mask_missing")
        rows.append({
            "case_id_opaque": case_opaque,
            "patient_id_opaque": r["patient_id_opaque"],
            "series_id_opaque": series_opaque,
            "sequence_role": r["sequence_role"],
            "split": r["subset"],
            "num_vertebrae": r.get("num_vertebrae"),
            "num_discs": r.get("num_discs"),
            "image_available": image_ok,
            "segmentation_available": mask_ok,
            "grading_available": None,  # populated in STEP 12
            "warnings": "; ".join(row_warn),
        })

    spider_dataset_inventory = pd.DataFrame(rows)

    n_images_found = len(image_stems)
    n_masks_found = len(mask_stems)
    n_paired = int((spider_dataset_inventory["image_available"] & spider_dataset_inventory["segmentation_available"]).sum())
    n_total_rows = len(spider_dataset_inventory)
    image_mask_pairing_summary = {
        "images_found_on_disk": n_images_found, "masks_found_on_disk": n_masks_found,
        "overview_rows": n_total_rows, "paired_rows": n_paired,
        "unpaired_rows": n_total_rows - n_paired,
    }
    GATE_E_image_mask_pairing = "PASS" if (n_total_rows > 0 and n_paired == n_total_rows) else "FAIL"
    if GATE_E_image_mask_pairing == "FAIL":
        warnings.append(f"IMAGE_MASK_PAIRING mismatch: {image_mask_pairing_summary}")
else:
    warnings.append("spider_dataset_inventory not built: GATE B did not PASS in this run.")

print("GATE_E_image_mask_pairing:", GATE_E_image_mask_pairing if GATE_B_dataset_structure_verified == "PASS" else "NOT_RUN")
print(json.dumps(image_mask_pairing_summary, indent=2) if image_mask_pairing_summary else "image_mask_pairing_summary: None")
spider_dataset_inventory.head()


GATE_E_image_mask_pairing: PASS
{
  "images_found_on_disk": 447,
  "masks_found_on_disk": 447,
  "overview_rows": 447,
  "paired_rows": 447,
  "unpaired_rows": 0
}


,case_id_opaque,patient_id_opaque,series_id_opaque,sequence_role,split,num_vertebrae,num_discs,image_available,segmentation_available,grading_available,warnings
0,3061e0d94b9f,6b86b273ff34,468b0ad79c39,t1,training,7,7,True,True,None,
1,549cb5cbe01f,6b86b273ff34,b7d987166b87,t2,training,7,7,True,True,None,
2,69d863bc64ce,4a44dc153642,cb0bf4dbb818,t1,training,7,7,True,True,None,
3,4111810b1b2e,4a44dc153642,374dba9bfceb,t2,training,7,7,True,True,None,
4,53e5a48c8d98,ad5736686512,e16e9bfaedae,t1,training,8,8,True,True,None,


## STEP 8b — `validation_t2_series_inventory.csv` — selección determinística de T2

Notebook 67 usó Sagittal T2. Se prioriza `*_t2` exacto; si no existe y hay `*_t2_SPACE`, se
registra aparte (no se selecciona manualmente). Se documenta la regla por paciente.


In [21]:
validation_t2_series_inventory = pd.DataFrame(columns=[
    "patient_id_opaque", "series_id_opaque", "sequence_role", "candidate_count_for_patient",
    "primary_series", "selection_reason", "warnings",
])

if len(spider_dataset_inventory) and len(split_inventory):
    val_patients = set(split_inventory.loc[split_inventory["split"].str.lower() == "validation", "patient_id_opaque"])
    rows = []
    for patient in sorted(val_patients):
        candidates = spider_dataset_inventory[
            (spider_dataset_inventory["patient_id_opaque"] == patient)
            & (spider_dataset_inventory["sequence_role"].isin(["t2", "t2_space"]))
        ]
        exact_t2 = candidates[candidates["sequence_role"] == "t2"]
        t2_space = candidates[candidates["sequence_role"] == "t2_space"]
        row_warn = []
        if len(exact_t2) >= 1:
            primary = exact_t2.iloc[0]
            reason = "exact_t2_preferred"
            if len(exact_t2) > 1:
                row_warn.append(f"{len(exact_t2)}_exact_t2_candidates_first_by_sorted_series_id_used")
        elif len(t2_space) >= 1:
            primary = t2_space.iloc[0]
            reason = "fallback_t2_space_no_exact_t2_available"
        else:
            primary = None
            reason = "no_t2_or_t2_space_candidate_found"
            row_warn.append("patient_excluded_from_smoke_test_pool")

        for _, cand in candidates.iterrows():
            rows.append({
                "patient_id_opaque": patient,
                "series_id_opaque": cand["series_id_opaque"],
                "sequence_role": cand["sequence_role"],
                "candidate_count_for_patient": len(candidates),
                "primary_series": bool(primary is not None and cand["series_id_opaque"] == primary["series_id_opaque"]),
                "selection_reason": reason,
                "warnings": "; ".join(row_warn),
            })
        if len(candidates) == 0:
            rows.append({
                "patient_id_opaque": patient, "series_id_opaque": None, "sequence_role": None,
                "candidate_count_for_patient": 0, "primary_series": False,
                "selection_reason": reason, "warnings": "; ".join(row_warn),
            })
    validation_t2_series_inventory = pd.DataFrame(rows)
else:
    warnings.append("validation_t2_series_inventory not built: dataset inventory or split_inventory unavailable.")

print("validation patients with a resolved primary T2 series:",
      int(validation_t2_series_inventory["primary_series"].sum()) if len(validation_t2_series_inventory) else 0)
validation_t2_series_inventory.head(10)


validation patients with a resolved primary T2 series: 38


,patient_id_opaque,series_id_opaque,sequence_role,candidate_count_for_patient,primary_series,selection_reason,warnings
0,043066daf210,543723a9240d,t2,2,True,exact_t2_preferred,
1,043066daf210,42f42d5f295c,t2_space,2,False,exact_t2_preferred,
2,114bd151f8fb,58d76dbe2d48,t2,2,True,exact_t2_preferred,
3,114bd151f8fb,28cfc992ede9,t2_space,2,False,exact_t2_preferred,
4,16dc368a89b4,ff97985d79fe,t2,1,True,exact_t2_preferred,
5,284de502c984,c7e7c675633f,t2,1,True,exact_t2_preferred,
6,2858dcd1057d,6f0846c0cadd,t2,1,True,exact_t2_preferred,
7,28dae7c8bde2,285cdb027505,t2,1,True,exact_t2_preferred,
8,3068430da9e4,45d8b95c090e,t2,1,True,exact_t2_preferred,
9,3346f2bbf6c3,793763191e99,t2,2,True,exact_t2_preferred,


## Validation cohort accounting (corrección) — total vs T2-evaluable

`39 validation patients` (del split público) **no** es lo mismo que `patients evaluables por este
pipeline`: solo los que tienen una serie T2 (`exact_t2` o fallback `t2_space`) resuelta pueden
correr por STEP 9. Se reporta explícitamente el desglose y la razón de exclusión por paciente --
nunca se imprime "full validation batch (N patients)" usando el total del split si ese total no
coincide con el evaluable.


In [22]:
validation_cohort_accounting = {
    "validation_patients_total": 0,
    "validation_patients_t2_evaluable": 0,
    "validation_patients_not_evaluable": 0,
    "not_evaluable_reasons": {},
}

if len(validation_t2_series_inventory):
    per_patient = validation_t2_series_inventory.groupby("patient_id_opaque").agg(
        evaluable=("primary_series", "any"),
        exclusion_reason=("selection_reason", "first"),
    ).reset_index()
    validation_patients_total = len(per_patient)
    evaluable_df = per_patient[per_patient["evaluable"]]
    not_evaluable_df = per_patient[~per_patient["evaluable"]]
    reason_counts = not_evaluable_df["exclusion_reason"].value_counts().to_dict() if len(not_evaluable_df) else {}
    validation_cohort_accounting = {
        "validation_patients_total": validation_patients_total,
        "validation_patients_t2_evaluable": int(len(evaluable_df)),
        "validation_patients_not_evaluable": int(len(not_evaluable_df)),
        "not_evaluable_reasons": {str(k): int(v) for k, v in reason_counts.items()},
        "not_evaluable_patient_ids_opaque": not_evaluable_df["patient_id_opaque"].tolist(),
    }
else:
    warnings.append("validation_cohort_accounting not computed: validation_t2_series_inventory unavailable in this run.")

print(json.dumps(validation_cohort_accounting, indent=2, default=str))
print()
print(f"validation_patients_total={validation_cohort_accounting['validation_patients_total']}, "
      f"validation_patients_t2_evaluable={validation_cohort_accounting['validation_patients_t2_evaluable']}, "
      f"validation_patients_not_evaluable={validation_cohort_accounting['validation_patients_not_evaluable']}")
print("Any future full validation batch attempts EXACTLY validation_patients_t2_evaluable cases; "
      "excluded patients are reported with reason, never silently dropped from the denominator.")


{
  "validation_patients_total": 39,
  "validation_patients_t2_evaluable": 38,
  "validation_patients_not_evaluable": 1,
  "not_evaluable_reasons": {
    "no_t2_or_t2_space_candidate_found": 1
  },
  "not_evaluable_patient_ids_opaque": [
    "65a699905c02"
  ]
}

validation_patients_total=39, validation_patients_t2_evaluable=38, validation_patients_not_evaluable=1
Any future full validation batch attempts EXACTLY validation_patients_t2_evaluable cases; excluded patients are reported with reason, never silently dropped from the denominator.


## Ground truth de INSTANCE DETECTION desde las máscaras (no desde `radiological_gradings`)

Labels de disco válidos: `>= 201` (esquema real verificado: vertebrae `1,2,3,...`, canal `100`,
IVD `201,202,203,...`). **No se convierte `201 -> L5-S1`** -- solo se registra identidad relativa
(`relative_instance_label`) y orden inferior→superior derivado del propio numbering del dataset.


In [23]:
DISC_LABEL_MIN = 201


def extract_gt_disc_instances_from_mask(mask_image: "sitk.Image") -> list[dict]:
    mask_array_raw = np.asarray(sitk.GetArrayFromImage(mask_image))  # (z, y, x) native order
    mask_array, mask_swapped = canonicalize_spider_array_with_swap_flag(mask_array_raw)
    labels = sorted(int(v) for v in np.unique(mask_array) if int(v) >= DISC_LABEL_MIN)

    instances = []
    for label in labels:
        binary = mask_array == label
        if binary.sum() == 0:
            continue
        idx = np.where(binary)
        # idx order follows mask_array's axes (row=0, col=1, slice=2 after canonicalization).
        centroid_row, centroid_col, centroid_slice = (float(np.mean(idx[0])), float(np.mean(idx[1])), float(np.mean(idx[2])))
        bbox = [int(idx[0].min()), int(idx[0].max()) + 1, int(idx[1].min()), int(idx[1].max()) + 1, int(idx[2].min()), int(idx[2].max()) + 1]
        centroid_xyz = canonical_index_to_physical_xyz(mask_image, centroid_row, centroid_col, centroid_slice, mask_swapped)
        instances.append({
            "mask_label": label,
            "relative_instance_label": None,  # filled below once all labels for this case are known (rank, inferior->superior)
            "centroid_index_row_col_slice": [centroid_row, centroid_col, centroid_slice],
            "centroid_physical_xyz": centroid_xyz.tolist(),
            "bbox": bbox,
            "voxel_count": int(binary.sum()),
            "present_in_fov": True,
        })

    # SPIDER numbers bottom-up: lower mask_label = more inferior. Rank ascending by mask_label.
    for rank, inst in enumerate(sorted(instances, key=lambda i: i["mask_label"]), start=1):
        inst["relative_instance_label"] = rank
    return instances


print("extract_gt_disc_instances_from_mask defined (labels >= 201; relative_instance_label = rank by mask_label, inferior->superior).")


extract_gt_disc_instances_from_mask defined (labels >= 201; relative_instance_label = rank by mask_label, inferior->superior).


## STEP 11 — Validar `num_discs` (mask vs overview) — `mask_overview_consistency.csv`


In [24]:
mask_overview_consistency_rows = []
warnings.append("mask_overview_consistency requires opening real mask .mha files -- populated per-case during the smoke test / full validation loop below, not as a separate full-cohort pass in this cell (avoids opening all 447 masks twice).")
mask_overview_consistency = pd.DataFrame(mask_overview_consistency_rows, columns=[
    "case_id_opaque", "overview_num_discs", "mask_disc_instance_count", "consistent",
])
print("mask_overview_consistency schema defined; rows populated during smoke test execution below.")


mask_overview_consistency schema defined; rows populated during smoke test execution below.


## STEP 12 — Auditar `radiological_gradings.csv` (NO usar para naming)

Se compara `overview.num_discs`, cantidad de mask IVD labels, y cantidad de grading IVD labels
únicos por paciente; se prueban transformaciones candidatas (`label-200`, `label-201`) y solo se
declara `GRADING_MASK_LABEL_MAPPING_VALIDATED` si una misma regla es consistente en **todos** los
casos aplicables. Independientemente del resultado, esto **nunca** se convierte en nivel
anatómico absoluto.


In [25]:
radiological_gradings_df = pd.DataFrame()
grading_mask_label_mapping_audit_rows = []
GRADING_MASK_LABEL_MAPPING_STATUS = "UNRESOLVED"

if SPIDER_AVAILABLE:
    grading_path = SPIDER_ROOT / "radiological_gradings.csv"
    if grading_path.is_file():
        radiological_gradings_df = pd.read_csv(grading_path)
        print("radiological_gradings.csv columns:", list(radiological_gradings_df.columns))
        print("rows:", len(radiological_gradings_df))
    else:
        warnings.append("radiological_gradings.csv not found under SPIDER_ROOT.")
else:
    warnings.append("radiological_gradings.csv audit skipped: SPIDER_AVAILABLE=False in this run.")

# The actual per-patient comparison against mask IVD labels requires opening masks (deferred to
# the smoke-test / full-validation loop, same reasoning as mask_overview_consistency, to avoid a
# separate full-cohort mask-opening pass). This cell defines the audit schema and the candidate
# transformations to test; GRADING_MASK_LABEL_MAPPING_STATUS is finalized after real cases are
# processed.
CANDIDATE_LABEL_TRANSFORMS = {"label_minus_200": lambda label: label - 200, "label_minus_201": lambda label: label - 201}
grading_mask_label_mapping_audit = pd.DataFrame(grading_mask_label_mapping_audit_rows, columns=[
    "patient_id_opaque", "overview_num_discs", "mask_ivd_label_count", "grading_ivd_label_count",
    "mask_label_min", "mask_label_max", "grading_label_min", "grading_label_max", "consistent_transform",
])
print("GRADING_MASK_LABEL_MAPPING_STATUS (provisional):", GRADING_MASK_LABEL_MAPPING_STATUS)
print("Regardless of outcome: NEVER converted into an absolute anatomical lumbar level in this notebook.")


radiological_gradings.csv columns: ['Patient', 'IVD label', 'Modic', 'UP endplate', 'LOW endplate', 'Spondylolisthesis', 'Disc herniation', 'Disc narrowing', 'Disc bulging', 'Pfirrman grade']
rows: 1520
GRADING_MASK_LABEL_MAPPING_STATUS (provisional): UNRESOLVED
Regardless of outcome: NEVER converted into an absolute anatomical lumbar level in this notebook.


## STEP 13 — Smoke test cohort (determinístico, solo VALIDATION, a nivel PACIENTE)


In [26]:
SMOKE_TEST_SEED = 2026
SMOKE_TEST_SIZE = 3


def select_smoke_test_cohort(patient_ids_opaque: list[str], seed: int = SMOKE_TEST_SEED, size: int = SMOKE_TEST_SIZE) -> list[str]:
    sorted_ids = sorted(patient_ids_opaque)
    rng = np.random.default_rng(seed)
    if len(sorted_ids) <= size:
        return sorted_ids
    indices = rng.choice(len(sorted_ids), size=size, replace=False)
    return sorted([sorted_ids[i] for i in sorted(indices)])


_synthetic_ids = [opaque_id(f"case_{i}") for i in range(10)]
assert select_smoke_test_cohort(_synthetic_ids) == select_smoke_test_cohort(list(reversed(_synthetic_ids))), "select_smoke_test_cohort self-test FAILED (order dependence)"
print("select_smoke_test_cohort: synthetic self-test PASSED.")

smoke_test_cohort = []
if len(validation_t2_series_inventory):
    eligible_patients = validation_t2_series_inventory.loc[validation_t2_series_inventory["primary_series"], "patient_id_opaque"].tolist()
    smoke_test_cohort = select_smoke_test_cohort(eligible_patients)
else:
    warnings.append("Smoke test cohort not selected: validation_t2_series_inventory unavailable in this run.")

print("smoke_test_cohort (validation patients, opaque IDs):", smoke_test_cohort)


select_smoke_test_cohort: synthetic self-test PASSED.
smoke_test_cohort (validation patients, opaque IDs): ['114bd151f8fb', '3068430da9e4', 'd029fa3a95e1']


## STEP 9 (real) — Pipeline completo V67_FROZEN_BASELINE

`load SPIDER T2 MHA → preprocesamiento congelado → checkpoint cf11... → inference → disc_group →
instance extraction (Notebook 67) → multi-slice consensus → ordered predicted instances → SPIDER
mask GT instances → physical centroid matching (SimpleITK) → Hungarian → metrics`. Antes de
Hungarian se audita paridad geométrica imagen/máscara (`size`/`spacing`/`origin`/`direction`);
si difiere inesperadamente: `FAIL` para ese caso, no se fuerza el matching.


In [27]:
def geometry_parity_audit(image: "sitk.Image", mask: "sitk.Image") -> dict:
    result = {
        "size_match": tuple(image.GetSize()) == tuple(mask.GetSize()),
        "spacing_match": np.allclose(image.GetSpacing(), mask.GetSpacing(), atol=1e-3),
        "origin_match": np.allclose(image.GetOrigin(), mask.GetOrigin(), atol=1e-3),
        "direction_match": np.allclose(image.GetDirection(), mask.GetDirection(), atol=1e-3),
    }
    result["geometry_parity_pass"] = all(result.values())
    return result


def run_v67_frozen_pipeline_on_case(image_path: Path, mask_path: Path, overview_num_discs) -> dict:
    if PREDICTION_TO_MHA_COORDINATE_ROUNDTRIP != "PASS":
        return {"status": "coordinate_roundtrip_not_validated"}

    image_sitk = sitk.ReadImage(str(image_path))
    mask_sitk = sitk.ReadImage(str(mask_path))

    parity = geometry_parity_audit(image_sitk, mask_sitk)
    if not parity["geometry_parity_pass"]:
        return {"status": "geometry_parity_failed", "geometry_parity": parity}

    image_array, image_swapped = canonicalize_spider_array_with_swap_flag(np.asarray(sitk.GetArrayFromImage(image_sitk)))
    n_slices = image_array.shape[SPIDER_SAGITTAL_AXIS]
    target_size = tuple(sagittal_runtime_meta["targetSize"])
    class_names_by_id = MODEL_REGISTRY.get("sagittal_spider", {}).get("class_names", {})
    disc_class_id = next((cid for cid, name in class_names_by_id.items() if name == "disc_group"), None)

    predicted_components = []  # list of dicts: slice_index, centroid_xyz, area_px
    for slice_index in range(n_slices):
        native_slice = np.take(image_array, slice_index, axis=SPIDER_SAGITTAL_AXIS)
        prepared = resize_image(robust_percentile_normalize(native_slice), target_size)
        tensor = torch.from_numpy(prepared[None, None]).float().to(DEVICE)
        with torch.inference_mode():
            logits = sagittal_model(tensor)
            probabilities = torch.softmax(logits, dim=1)[0]
        prediction = torch.argmax(probabilities, dim=0).cpu().numpy().astype(np.uint8)
        confidence = torch.max(probabilities, dim=0).values.cpu().numpy().astype(np.float32)

        disc_mask = prediction == disc_class_id if disc_class_id is not None else np.zeros_like(prediction, dtype=bool)
        for component in connected_instances(disc_mask):
            geom = component_geometry(component)
            # Inverse of resize_image: model-grid pixel -> native (pre-resize) canonical pixel.
            row_native = geom["centroid_row"] * (native_slice.shape[0] / target_size[0])
            col_native = geom["centroid_col"] * (native_slice.shape[1] / target_size[1])
            # canonical_index_to_physical_xyz uses `image_sitk` (the ORIGINAL, pre-preprocessing
            # image) directly -- the predicted centroid is expressed in the original image's
            # physical geometry, not the model/resized grid or the mask's geometry.
            centroid_xyz = canonical_index_to_physical_xyz(image_sitk, row_native, col_native, float(slice_index), image_swapped)
            # Self-check: physical point round-trips back to the native canonical index we started
            # from, confirming the coordinate is anchored to image_sitk's own geometry.
            i_native, j_native, k_native = image_sitk.TransformPhysicalPointToContinuousIndex(tuple(centroid_xyz.tolist()))
            row_back, col_back, slice_back = native_sitk_index_to_canonical_index(i_native, j_native, k_native, image_swapped)
            assert abs(row_back - row_native) < 1e-3 and abs(col_back - col_native) < 1e-3 and abs(slice_back - slice_index) < 1e-3, \
                "predicted centroid does not round-trip to original image geometry (PREDICTION_TO_MHA_COORDINATE_ROUNDTRIP violated at runtime)"
            predicted_components.append({
                "slice_index": slice_index, "centroid_xyz": centroid_xyz, "area_px": geom["area_px"],
                "mean_confidence": float(confidence.mean()), "border_touch": geom["border_touch"],
                "prediction_geometry_reference": "original_image_sitk",
            })

    # Multi-slice consensus.
    consensus_instances = []
    if predicted_components:
        groups = consensus_union_find([c["centroid_xyz"] for c in predicted_components], distance_threshold_mm=15.0)
        for indices in groups:
            members = [predicted_components[i] for i in indices]
            centroids = np.stack([m["centroid_xyz"] for m in members])
            mean_centroid = centroids.mean(axis=0)
            spread = float(np.max(np.linalg.norm(centroids - mean_centroid, axis=1))) if len(members) > 1 else 0.0
            confidence_val = compute_instance_confidence(len(members), spread, float(np.mean([m["mean_confidence"] for m in members])))
            consensus_instances.append({"centroid_xyz": mean_centroid, "supporting_slice_count": len(members), "instance_confidence": confidence_val})

    # Predicted relative ordering (arbitrary consistent axis, not mapped to cranial/caudal).
    predicted_ordering = []
    if len(consensus_instances) >= 2:
        axis = spine_axis_from_points(np.stack([c["centroid_xyz"] for c in consensus_instances]))
        ref = consensus_instances[0]["centroid_xyz"]
        for c in consensus_instances:
            c["position_along_axis_mm"] = float(np.dot(c["centroid_xyz"] - ref, axis))
        consensus_instances.sort(key=lambda c: c["position_along_axis_mm"])
    predicted_ordering = [{"rank": i + 1, **{k: v for k, v in c.items() if k != "centroid_xyz"}} for i, c in enumerate(consensus_instances)]

    # Ground truth from mask.
    gt_instances = extract_gt_disc_instances_from_mask(mask_sitk)
    mask_disc_instance_count = len(gt_instances)

    # Matching.
    pred_centroids = [c["centroid_xyz"] for c in consensus_instances]
    gt_centroids = [np.array(g["centroid_physical_xyz"]) for g in gt_instances]
    match_result = match_predictions_to_ground_truth(pred_centroids, gt_centroids)
    metrics = instance_detection_metrics(match_result, len(pred_centroids), len(gt_centroids))
    ordering_metrics = compute_relative_ordering_metrics(match_result, predicted_ordering, gt_instances)

    return {
        "status": "executed",
        "geometry_parity": parity,
        "gt_disc_count": mask_disc_instance_count,
        "overview_num_discs": overview_num_discs,
        "mask_overview_consistent": (overview_num_discs is not None and int(overview_num_discs) == mask_disc_instance_count),
        "predicted_instance_count": len(consensus_instances),
        "predicted_ordering": predicted_ordering,
        "gt_relative_labels": [g["relative_instance_label"] for g in gt_instances],
        "gt_mask_labels": [g["mask_label"] for g in gt_instances],
        "match_result": match_result,
        "metrics": metrics,
        "ordering_metrics": ordering_metrics,
    }


print("run_v67_frozen_pipeline_on_case defined (real implementation, no extension-point stub).")


run_v67_frozen_pipeline_on_case defined (real implementation, no extension-point stub).


## STEP 9 (ejecución) — Smoke test real sobre 3 casos de validation (GATE D)

`GATE D = PASS` si los 3/3 casos ejecutan sin fallo de runtime, con el checkpoint correcto,
geometría imagen/máscara válida, GT extraído, predicciones producidas y métricas de Hungarian
calculadas. **PASS no significa alta accuracy** -- se separa `EXECUTION PASS` de
`MODEL PERFORMANCE`, sin threshold de performance todavía.


In [28]:
smoke_test_results_rows = []
mask_overview_consistency_rows = []
grading_audit_rows = []
GATE_D_smoke_test = "FAIL"

if SPIDER_AVAILABLE and GATE_A_checkpoint_identity == "PASS" and GATE_E_image_mask_pairing == "PASS" and smoke_test_cohort:
    images_dir = SPIDER_ROOT / "images" / "images"
    masks_dir = SPIDER_ROOT / "masks" / "masks"
    grading_by_patient = {}
    if len(radiological_gradings_df) and "Patient" in radiological_gradings_df.columns:
        for patient_raw, group in radiological_gradings_df.groupby("Patient"):
            grading_by_patient[opaque_id(str(patient_raw))] = group

    executed_ok = 0
    for patient in smoke_test_cohort:
        primary_row = validation_t2_series_inventory[
            (validation_t2_series_inventory["patient_id_opaque"] == patient) & (validation_t2_series_inventory["primary_series"])
        ]
        if not len(primary_row):
            smoke_test_results_rows.append({"patient_id_opaque": patient, "status": "no_primary_t2_series"})
            continue
        series_row = primary_row.iloc[0]
        inv_row = spider_dataset_inventory[spider_dataset_inventory["series_id_opaque"] == series_row["series_id_opaque"]].iloc[0]
        stem = None
        for _, r in overview_df.iterrows():
            if opaque_id(str(r["new_file_name"])) == series_row["series_id_opaque"]:
                stem = str(r["new_file_name"])
                break
        if stem is None:
            smoke_test_results_rows.append({"patient_id_opaque": patient, "status": "series_stem_not_resolved"})
            continue

        image_path = images_dir / f"{stem}.mha"
        mask_path = masks_dir / f"{stem}.mha"
        try:
            result = run_v67_frozen_pipeline_on_case(image_path, mask_path, inv_row.get("num_discs"))
        except Exception as exc:
            smoke_test_results_rows.append({
                "patient_id_opaque": patient, "series_id_opaque": series_row["series_id_opaque"],
                "sequence": series_row["sequence_role"], "status": f"runtime_error: {exc}",
            })
            continue

        if result["status"] != "executed":
            smoke_test_results_rows.append({
                "patient_id_opaque": patient, "series_id_opaque": series_row["series_id_opaque"],
                "sequence": series_row["sequence_role"], "status": result["status"],
                "geometry_parity": result.get("geometry_parity"),
            })
            continue

        executed_ok += 1
        m = result["metrics"]
        om = result["ordering_metrics"]
        smoke_test_results_rows.append({
            "patient_id_opaque": patient, "series_id_opaque": series_row["series_id_opaque"], "sequence": series_row["sequence_role"],
            "gt_disc_count": result["gt_disc_count"], "predicted_instance_count": result["predicted_instance_count"],
            "matched_count": len(result["match_result"]["matches"]), "false_positive_count": m["false_positive_count"],
            "false_negative_count": m["false_negative_count"], "precision": m["disc_detection_precision"],
            "recall": m["disc_detection_recall"], "f1": m["disc_detection_f1"],
            "centroid_error_mean_mm": m["centroid_error_mean_mm"], "centroid_error_median_mm": m["centroid_error_median_mm"],
            "centroid_error_max_mm": m["centroid_error_max_mm"], "gt_relative_labels": result["gt_relative_labels"],
            "predicted_ordering": result["predicted_ordering"], "matching_pairs": result["match_result"]["matches"],
            "matched_pair_count": om["matched_pair_count"], "relative_order_pair_accuracy": om["relative_order_pair_accuracy"],
            "relative_order_monotonic": om["relative_order_monotonic"], "relative_order_direction": om["relative_order_direction"],
            "relative_order_inversion_count": om["relative_order_inversion_count"],
            "relative_order_crossing_count": om["relative_order_crossing_count"],
            "spearman_rank_correlation": om["spearman_rank_correlation"],
            "status": "executed",
        })

        mask_overview_consistency_rows.append({
            "case_id_opaque": inv_row["case_id_opaque"], "overview_num_discs": inv_row.get("num_discs"),
            "mask_disc_instance_count": result["gt_disc_count"], "consistent": result["mask_overview_consistent"],
        })

        patient_grading = grading_by_patient.get(patient)
        ivd_col = next((c for c in radiological_gradings_df.columns if "ivd" in c.lower() and "label" in c.lower()), None)
        if patient_grading is not None and ivd_col is not None:
            grading_labels = sorted(int(v) for v in patient_grading[ivd_col].dropna().unique())
            mask_labels_this_case = sorted(result["gt_mask_labels"])
            consistent_transforms = []
            for transform_name, transform_fn in CANDIDATE_LABEL_TRANSFORMS.items():
                transformed_mask_labels = sorted(transform_fn(lbl) for lbl in mask_labels_this_case)
                if transformed_mask_labels == grading_labels:
                    consistent_transforms.append(transform_name)
            grading_audit_rows.append({
                "patient_id_opaque": patient, "overview_num_discs": inv_row.get("num_discs"),
                "mask_ivd_label_count": len(mask_labels_this_case), "grading_ivd_label_count": len(grading_labels),
                "mask_label_min": min(mask_labels_this_case) if mask_labels_this_case else None,
                "mask_label_max": max(mask_labels_this_case) if mask_labels_this_case else None,
                "grading_label_min": min(grading_labels) if grading_labels else None,
                "grading_label_max": max(grading_labels) if grading_labels else None,
                "consistent_transform": consistent_transforms if consistent_transforms else "NO_CANDIDATE_TRANSFORM_MATCHED",
            })

    GATE_D_smoke_test = "PASS" if (len(smoke_test_cohort) > 0 and executed_ok == len(smoke_test_cohort)) else "FAIL"
else:
    warnings.append("Smoke test NOT executed: SPIDER unavailable, checkpoint invalid, image/mask pairing invalid, or empty cohort in this run.")


def aggregate_relative_ordering_validation(rows: list[dict]) -> str:
    # GATE F (detection) is intentionally NOT reused here -- this is a distinct sub-audit.
    assessable = [r for r in rows if r.get("status") == "executed" and r.get("matched_pair_count", 0) >= 2]
    if not assessable:
        return "UNRESOLVED"
    monotonic_flags = [bool(r["relative_order_monotonic"]) for r in assessable]
    if all(monotonic_flags):
        return "PASS"
    if any(monotonic_flags):
        return "PARTIAL"
    return "FAIL"


RELATIVE_ORDERING_VALIDATION = aggregate_relative_ordering_validation(smoke_test_results_rows)
print("RELATIVE_ORDERING_VALIDATION (smoke-test cohort, distinct from GATE_F detection):", RELATIVE_ORDERING_VALIDATION)

smoke_test_results = pd.DataFrame(smoke_test_results_rows)
mask_overview_consistency = pd.DataFrame(mask_overview_consistency_rows) if mask_overview_consistency_rows else mask_overview_consistency
grading_mask_label_mapping_audit = pd.DataFrame(grading_audit_rows) if grading_audit_rows else grading_mask_label_mapping_audit

print("GATE_D_smoke_test:", GATE_D_smoke_test if (SPIDER_AVAILABLE and smoke_test_cohort) else "NOT_RUN")
smoke_test_results


RELATIVE_ORDERING_VALIDATION (smoke-test cohort, distinct from GATE_F detection): PASS
GATE_D_smoke_test: PASS


,patient_id_opaque,series_id_opaque,sequence,gt_disc_count,predicted_instance_count,matched_count,false_positive_count,false_negative_count,precision,recall,...,predicted_ordering,matching_pairs,matched_pair_count,relative_order_pair_accuracy,relative_order_monotonic,relative_order_direction,relative_order_inversion_count,relative_order_crossing_count,spearman_rank_correlation,status
0,114bd151f8fb,58d76dbe2d48,t2,9,9,9,0,0,1.000,1.0,...,"[{'rank': 1, 'supporting_slice_count': 9, 'ins...","[{'pred_index': 0, 'gt_index': 8, 'centroid_er...",9,1.0,True,ORDER_DIRECTION_UNRESOLVED,0,0,-1.0,executed
1,3068430da9e4,45d8b95c090e,t2,7,8,7,1,0,0.875,1.0,...,"[{'rank': 1, 'supporting_slice_count': 13, 'in...","[{'pred_index': 0, 'gt_index': 0, 'centroid_er...",7,1.0,True,ORDER_DIRECTION_UNRESOLVED,0,0,1.0,executed
2,d029fa3a95e1,87ece9cc6462,t2,8,8,8,0,0,1.000,1.0,...,"[{'rank': 1, 'supporting_slice_count': 17, 'in...","[{'pred_index': 0, 'gt_index': 0, 'centroid_er...",8,1.0,True,ORDER_DIRECTION_UNRESOLVED,0,0,1.0,executed


## Finalizar `GRADING_MASK_LABEL_MAPPING_VALIDATED` / `mask_overview_consistency`

Solo se declara `GRADING_MASK_LABEL_MAPPING_VALIDATED` si la **misma** transformación es
consistente en todos los casos con evidencia disponible (no se elige "porque parece lógica").


In [29]:
if len(grading_mask_label_mapping_audit):
    all_transform_sets = [
        set(row) if isinstance(row, list) else set() for row in grading_mask_label_mapping_audit["consistent_transform"]
    ]
    common_transforms = set.intersection(*all_transform_sets) if all_transform_sets and all(all_transform_sets) else set()
    GRADING_MASK_LABEL_MAPPING_STATUS = f"VALIDATED:{sorted(common_transforms)}" if common_transforms else "UNRESOLVED"
else:
    GRADING_MASK_LABEL_MAPPING_STATUS = "UNRESOLVED"

print("GRADING_MASK_LABEL_MAPPING_STATUS (final, this run's cohort only):", GRADING_MASK_LABEL_MAPPING_STATUS)
print("Reminder: even if validated, this mapping is NEVER converted into an absolute anatomical lumbar level.")

MASK_OVERVIEW_CONSISTENCY_RATE = float(mask_overview_consistency["consistent"].mean()) if len(mask_overview_consistency) else None
print("mask_overview_consistency rate (this run's cohort):", MASK_OVERVIEW_CONSISTENCY_RATE)
mask_overview_consistency


GRADING_MASK_LABEL_MAPPING_STATUS (final, this run's cohort only): VALIDATED:['label_minus_200']
Reminder: even if validated, this mapping is NEVER converted into an absolute anatomical lumbar level.
mask_overview_consistency rate (this run's cohort): 1.0


,case_id_opaque,overview_num_discs,mask_disc_instance_count,consistent
0,dec869a3ee93,9,9,True
1,9d8abe7d513c,7,7,True
2,c1d72e3e881f,8,8,True


## STEP 19 — FOV analysis (GATE G)

Notebook 67 produjo 7 candidatos sobre el DICOM externo real. Aquí se usa SPIDER para responder
cuántos discos aparecen realmente en distintos FOV -- **sin concluir de antemano** que 7 es error.

**Corrección metodológica (esta revisión):** la distribución previa mezclaba las **87 series**
(T1/T2/T2_SPACE, con posibles duplicados por paciente) del split de validation bajo el nombre
genérico `num_discs distribution`. Se renombra explícitamente a
`validation_series_num_discs_distribution` para dejar claro su alcance real (por serie, no por
paciente), y se agrega `validation_primary_t2_num_discs_distribution`, que usa **exactamente una**
serie T2 primaria por paciente evaluable (la misma selección determinística de STEP 8b) -- esta es
la distribución que debe compararse contra el pipeline, no la mezcla de series.


In [30]:
GATE_G_fov_analysis = "FAIL"
validation_gt_disc_count_distribution = pd.Series(dtype=float)
validation_series_num_discs_distribution = pd.Series(dtype=float)
validation_primary_t2_num_discs_distribution = pd.Series(dtype=float)

if len(overview_df) and len(split_inventory):
    val_patients = set(split_inventory.loc[split_inventory["split"].str.lower() == "validation", "patient_id_opaque"])
    df = overview_df.copy()
    df["patient_id_raw"] = df["new_file_name"].apply(derive_patient_id_raw)
    df = df.dropna(subset=["patient_id_raw"])
    df["patient_id_opaque"] = df["patient_id_raw"].apply(opaque_id)
    val_rows = df[df["patient_id_opaque"].isin(val_patients)]
    validation_series_num_discs_distribution = val_rows["num_discs"].value_counts().sort_index()
    print("validation_series_num_discs_distribution (ALL validation series, T1+T2+T2_SPACE, may include >1 per patient):")
    print(validation_series_num_discs_distribution)

    if len(validation_t2_series_inventory):
        primary_rows = validation_t2_series_inventory[validation_t2_series_inventory["primary_series"]]
        primary_with_counts = primary_rows.merge(
            spider_dataset_inventory[["series_id_opaque", "num_discs"]], on="series_id_opaque", how="left",
        )
        validation_primary_t2_num_discs_distribution = primary_with_counts["num_discs"].value_counts().sort_index()
        print()
        print("validation_primary_t2_num_discs_distribution (exactly ONE primary T2 per T2-evaluable patient -- PRIMARY comparison distribution):")
        print(validation_primary_t2_num_discs_distribution)

    if len(mask_overview_consistency):
        validation_gt_disc_count_distribution = mask_overview_consistency["mask_disc_instance_count"].value_counts().sort_index()
        print()
        print("mask GT disc count distribution (this run's smoke-test cohort only):")
        print(validation_gt_disc_count_distribution)
        GATE_G_fov_analysis = "PASS"
    else:
        warnings.append("GT mask disc count distribution not available yet -- only overview.csv distribution computed (full validation batch would populate this fully).")
        GATE_G_fov_analysis = "PARTIAL"
else:
    warnings.append("FOV analysis NOT RUN: overview.csv/split_inventory unavailable in this run.")

print()
print("GATE_G_fov_analysis:", GATE_G_fov_analysis if len(overview_df) else "NOT_RUN")
print("Reminder: Notebook 67's real-DICOM 7 candidates are NOT concluded to be an error by this analysis.")


validation_series_num_discs_distribution (ALL validation series, T1+T2+T2_SPACE, may include >1 per patient):
num_discs
3     1
6    28
7    21
8    22
9    15
Name: count, dtype: int64

validation_primary_t2_num_discs_distribution (exactly ONE primary T2 per T2-evaluable patient -- PRIMARY comparison distribution):
num_discs
3     1
6    14
7     9
8     9
9     5
Name: count, dtype: int64

mask GT disc count distribution (this run's smoke-test cohort only):
mask_disc_instance_count
7    1
8    1
9    1
Name: count, dtype: int64

GATE_G_fov_analysis: PASS
Reminder: Notebook 67's real-DICOM 7 candidates are NOT concluded to be an error by this analysis.


## STEP 20 — Absolute anatomical anchor (GATE H) y level naming (GATE I) — corrección crítica

SPIDER **no** provee ground truth de nivel anatómico absoluto (`L1-L2...L5-S1`) -- solo identidad
relativa de instancia (bottom-up). Por lo tanto `GATE H` y `GATE I` usan
`UNAVAILABLE_FROM_SPIDER_REFERENCE` / `UNAVAILABLE`, que es un **resultado metodológico válido**,
no un `FAIL`. No se genera ninguna confusion matrix `L1-L2...L5-S1` sin GT real.


In [31]:
def anchor_method_a_inferior_disc_context() -> dict:
    return {"anchor_available": False, "anchor_confidence": 0.0, "anchor_reason": "no_dedicated_sacral_class_in_checkpoint", "predicted_target_window": None, "warnings": []}


def anchor_method_b_vertebral_sequence_context(vertebra_instances_separable: bool) -> dict:
    if not vertebra_instances_separable:
        return {"anchor_available": False, "anchor_confidence": 0.0, "anchor_reason": "vertebral_instances_not_separable", "predicted_target_window": None, "warnings": []}
    return {"anchor_available": False, "anchor_confidence": 0.0, "anchor_reason": "spider_provides_relative_vertebral_instances_only_not_absolute_level", "predicted_target_window": None, "warnings": []}


def anchor_method_c_disc_spacing_pattern(distance_to_next_mm: list[float]) -> dict:
    if len(distance_to_next_mm) < 2:
        return {"anchor_available": False, "anchor_confidence": 0.0, "anchor_reason": "insufficient_spacing_samples", "predicted_target_window": None, "warnings": []}
    cv = float(np.std(distance_to_next_mm) / np.mean(distance_to_next_mm)) if np.mean(distance_to_next_mm) else 999.0
    return {
        "anchor_available": True, "anchor_confidence": round(max(0.0, 1.0 - cv), 4),
        "anchor_reason": f"spacing_coefficient_of_variation={cv:.3f} (informational only)",
        "predicted_target_window": None, "warnings": ["spacing_regularity_alone_cannot_resolve_absolute_target_window"],
    }


def anchor_method_d_combined(results: list[dict]) -> dict:
    resolved = [r for r in results if r["anchor_available"] and r["predicted_target_window"]]
    if not resolved:
        return {"anchor_available": False, "anchor_confidence": 0.0, "anchor_reason": "no_component_anchor_resolved_absolute_target_window", "predicted_target_window": None, "warnings": []}
    return {"anchor_available": True, "anchor_confidence": 0.5, "anchor_reason": "combined_component_anchors", "predicted_target_window": resolved[0]["predicted_target_window"], "warnings": []}


GATE_H_absolute_anatomical_anchor = "UNAVAILABLE_FROM_DATASET_REFERENCE"
GATE_I_absolute_level_naming_metrics = "UNAVAILABLE"

anchor_a = anchor_method_a_inferior_disc_context()
anchor_b = anchor_method_b_vertebral_sequence_context(vertebra_instances_separable=False)
anchor_c = anchor_method_c_disc_spacing_pattern([])
anchor_d = anchor_method_d_combined([anchor_a, anchor_b, anchor_c])
anchor_experiment_rows = [
    {"method": "A_inferior_disc_sacral_context", **anchor_a},
    {"method": "B_vertebral_sequence_context", **anchor_b},
    {"method": "C_disc_spacing_pattern", **anchor_c},
    {"method": "D_combined", **anchor_d},
]

print(pd.DataFrame(anchor_experiment_rows))
print()
print("GATE_H_absolute_anatomical_anchor:", GATE_H_absolute_anatomical_anchor)
print("GATE_I_absolute_level_naming_metrics:", GATE_I_absolute_level_naming_metrics)
print("No L1-L2..L5-S1 confusion matrix generated: no absolute ground truth exists in this dataset/run.")


                           method  anchor_available  anchor_confidence  \
0  A_inferior_disc_sacral_context             False                0.0   
1    B_vertebral_sequence_context             False                0.0   
2          C_disc_spacing_pattern             False                0.0   
3                      D_combined             False                0.0   

                                       anchor_reason predicted_target_window  \
0            no_dedicated_sacral_class_in_checkpoint                    None   
1                  vertebral_instances_not_separable                    None   
2                       insufficient_spacing_samples                    None   
3  no_component_anchor_resolved_absolute_target_w...                    None   

  warnings  
0       []  
1       []  
2       []  
3       []  

GATE_H_absolute_anatomical_anchor: UNAVAILABLE_FROM_DATASET_REFERENCE
GATE_I_absolute_level_naming_metrics: UNAVAILABLE
No L1-L2..L5-S1 confusion matrix generated

## STEP 10 — `RUN_FULL_VALIDATION` (control manual — corregido: cohort exacto + relative ordering)

`RUN_FULL_VALIDATION=False` en este run (no se ejecuta todavía, por instrucción explícita). El
batch, cuando se active, intenta **exactamente** `validation_cohort_accounting["validation_patients_t2_evaluable"]`
casos (no el total del split), reporta los excluidos con razón, y agrega las métricas de relative
ordering (STEP 9) a cada caso, no solo detección.


In [41]:
RUN_FULL_VALIDATION = True  # <-- change to True manually in Colab after reviewing smoke test AND this microcorrection

frozen_baseline_case_metrics = pd.DataFrame()
frozen_baseline_summary = None

if not RUN_FULL_VALIDATION:
    print(f"RUN_FULL_VALIDATION=False: full validation batch NOT executed. "
          f"Would attempt exactly {validation_cohort_accounting['validation_patients_t2_evaluable']} "
          f"T2-evaluable patients (of {validation_cohort_accounting['validation_patients_total']} total validation patients; "
          f"{validation_cohort_accounting['validation_patients_not_evaluable']} excluded, see not_evaluable_reasons). "
          "User will bring smoke test results for review first.")
elif not (SPIDER_AVAILABLE and GATE_D_smoke_test == "PASS"):
    print("RUN_FULL_VALIDATION=True but prerequisites not met (SPIDER unavailable or smoke test did not PASS) -- not proceeding.")
else:
    val_patients_all = validation_t2_series_inventory.loc[validation_t2_series_inventory["primary_series"], "patient_id_opaque"].tolist()
    excluded_no_t2 = validation_cohort_accounting.get("not_evaluable_patient_ids_opaque", [])
    images_dir = SPIDER_ROOT / "images" / "images"
    masks_dir = SPIDER_ROOT / "masks" / "masks"
    rows = []
    for patient in sorted(val_patients_all):
        series_row = validation_t2_series_inventory[
            (validation_t2_series_inventory["patient_id_opaque"] == patient) & (validation_t2_series_inventory["primary_series"])
        ].iloc[0]
        inv_row = spider_dataset_inventory[spider_dataset_inventory["series_id_opaque"] == series_row["series_id_opaque"]].iloc[0]
        stem = next((str(r["new_file_name"]) for _, r in overview_df.iterrows() if opaque_id(str(r["new_file_name"])) == series_row["series_id_opaque"]), None)
        if stem is None:
            rows.append({"patient_id_opaque": patient, "status": "series_stem_not_resolved"})
            continue
        try:
            result = run_v67_frozen_pipeline_on_case(images_dir / f"{stem}.mha", masks_dir / f"{stem}.mha", inv_row.get("num_discs"))
        except Exception as exc:
            rows.append({"patient_id_opaque": patient, "status": f"runtime_error: {exc}"})
            continue
        if result["status"] != "executed":
            rows.append({"patient_id_opaque": patient, "status": result["status"]})
            continue
        m = result["metrics"]
        om = result["ordering_metrics"]
        rows.append({
            "patient_id_opaque": patient, "gt_disc_count": result["gt_disc_count"],
            "predicted_instance_count": result["predicted_instance_count"],
            "matched_count": len(result["match_result"]["matches"]), "precision": m["disc_detection_precision"],
            "recall": m["disc_detection_recall"], "f1": m["disc_detection_f1"],
            "centroid_error_mean_mm": m["centroid_error_mean_mm"],
            "matched_pair_count": om["matched_pair_count"], "relative_order_pair_accuracy": om["relative_order_pair_accuracy"],
            "relative_order_monotonic": om["relative_order_monotonic"], "relative_order_direction": om["relative_order_direction"],
            "relative_order_inversion_count": om["relative_order_inversion_count"],
            "relative_order_crossing_count": om["relative_order_crossing_count"],
            "spearman_rank_correlation": om["spearman_rank_correlation"],
            "status": "executed",
        })
    for patient in excluded_no_t2:
        rows.append({"patient_id_opaque": patient, "status": "excluded_no_t2"})

    frozen_baseline_case_metrics = pd.DataFrame(rows)
    executed = frozen_baseline_case_metrics[frozen_baseline_case_metrics["status"] == "executed"] if len(frozen_baseline_case_metrics) else pd.DataFrame()
    failed = frozen_baseline_case_metrics[
        ~frozen_baseline_case_metrics["status"].isin(["executed", "excluded_no_t2"])
    ] if len(frozen_baseline_case_metrics) else pd.DataFrame()
    ordering_assessable = executed[executed["matched_pair_count"].fillna(0) >= 2] if len(executed) else pd.DataFrame()

    frozen_baseline_summary = {
        "cases_total_validation": validation_cohort_accounting["validation_patients_total"],
        "cases_t2_evaluable": validation_cohort_accounting["validation_patients_t2_evaluable"],
        "cases_attempted": len(val_patients_all),
        "cases_executed": len(executed),
        "cases_failed": len(failed),
        "cases_excluded_no_t2": len(excluded_no_t2),
        "mean_precision": float(executed["precision"].mean()) if len(executed) else None,
        "mean_recall": float(executed["recall"].mean()) if len(executed) else None,
        "mean_f1": float(executed["f1"].mean()) if len(executed) else None,
        "median_f1": float(executed["f1"].median()) if len(executed) else None,
        "mean_centroid_error_mm": float(executed["centroid_error_mean_mm"].mean()) if len(executed) else None,
        "median_centroid_error_mm": float(executed["centroid_error_mean_mm"].median()) if len(executed) else None,
        "relative_order_monotonic_rate": float(ordering_assessable["relative_order_monotonic"].mean()) if len(ordering_assessable) else None,
        "relative_order_direction_resolved_rate": float((ordering_assessable["relative_order_direction"] != "ORDER_DIRECTION_UNRESOLVED").mean()) if len(ordering_assessable) else None,
        "mean_spearman_rank_correlation": float(ordering_assessable["spearman_rank_correlation"].astype(float).mean()) if len(ordering_assessable) else None,
    }
    print(json.dumps(frozen_baseline_summary, indent=2, default=str))

print("GATE (full validation) status:", "NOT_RUN" if not RUN_FULL_VALIDATION else ("PASS" if frozen_baseline_summary else "FAIL"))


{
  "cases_total_validation": 39,
  "cases_t2_evaluable": 38,
  "cases_attempted": 38,
  "cases_executed": 38,
  "cases_failed": 0,
  "cases_excluded_no_t2": 1,
  "mean_precision": 0.9506996658312448,
  "mean_recall": 1.0,
  "mean_f1": 0.9721906576395739,
  "median_f1": 1.0,
  "mean_centroid_error_mm": 1.7523566724391513,
  "median_centroid_error_mm": 1.5368929128025899,
  "relative_order_monotonic_rate": 1.0,
  "relative_order_direction_resolved_rate": 0.0,
  "mean_spearman_rank_correlation": 0.05263157894736842
}
GATE (full validation) status: PASS


## STEP 13 — `TEST_SPLIT_LOCKED` (sin cambios; hidden test permanece fuera de alcance)


In [42]:
TEST_SPLIT_LOCKED = True
RUN_FINAL_TEST = False  # intentionally ignored below; see TEST_SPLIT_LOCKED
PUBLIC_TEST_AVAILABILITY = "HIDDEN_EXTERNAL_NOT_AVAILABLE"

if TEST_SPLIT_LOCKED:
    print("FINAL TEST RESERVED AFTER VALIDATION DESIGN FREEZE")
    print("TEST_SPLIT_LOCKED=True -- refusing to run test split regardless of RUN_FINAL_TEST value.")
    print("public_test_availability:", PUBLIC_TEST_AVAILABILITY, "-- the SPIDER hidden challenge test is out of scope; not accessed.")
    if RUN_FINAL_TEST:
        warnings.append("RUN_FINAL_TEST was set to True but TEST_SPLIT_LOCKED overrides it -- test was NOT executed.")
else:
    raise RuntimeError("TEST_SPLIT_LOCKED must remain True in Notebook 67A.")

test_patients_used = 0
print("test_patients_used:", test_patients_used)


FINAL TEST RESERVED AFTER VALIDATION DESIGN FREEZE
TEST_SPLIT_LOCKED=True -- refusing to run test split regardless of RUN_FINAL_TEST value.
public_test_availability: HIDDEN_EXTERNAL_NOT_AVAILABLE -- the SPIDER hidden challenge test is out of scope; not accessed.
test_patients_used: 0


## Quality gates A–J (revisados)

Sub-audits: preprocessing parity ahora se reporta como 4 componentes independientes
(`AXIS_CANONICALIZATION_PARITY`, `RESIZE_PARITY`, `INTENSITY_PREPROCESSING_PARITY`,
`MODEL_INPUT_SHAPE_PARITY`) en vez de un único `PASS` agregado -- ver sección anterior para la
fuente concreta de cada uno. `MHA geometry parity` = por-caso en el smoke test,
`prediction_to_mha_coordinate_roundtrip` = `PREDICTION_TO_MHA_COORDINATE_ROUNDTRIP` (gate crítico:
si no es `PASS`, el smoke test no produce `centroid_error_mm`), `overview-mask disc-count
consistency` = `mask_overview_consistency`, `grading-mask label mapping audit` =
`GRADING_MASK_LABEL_MAPPING_STATUS`.


In [43]:
gate_status: dict[str, str] = {
    "GATE_A_checkpoint_identity": GATE_A_checkpoint_identity,
    "GATE_B_spider_structure": GATE_B_dataset_structure_verified if SPIDER_AVAILABLE else "NOT_RUN",
    "GATE_C_public_train_validation_leakage": GATE_C_split_leakage_audit if GATE_B_dataset_structure_verified == "PASS" else "NOT_RUN",
    "GATE_D_smoke_test_execution": GATE_D_smoke_test if (SPIDER_AVAILABLE and smoke_test_cohort) else "NOT_RUN",
    "GATE_E_image_mask_exact_pairing": GATE_E_image_mask_pairing if GATE_B_dataset_structure_verified == "PASS" else "NOT_RUN",
    "GATE_F_instance_gt_extraction_metrics": "PASS" if GATE_D_smoke_test == "PASS" else ("NOT_RUN" if not (SPIDER_AVAILABLE and smoke_test_cohort) else "FAIL"),
    "GATE_G_fov_analysis": GATE_G_fov_analysis if len(overview_df) else "NOT_RUN",
    "GATE_H_absolute_anatomical_anchor": GATE_H_absolute_anatomical_anchor,
    "GATE_I_absolute_level_naming_metrics": GATE_I_absolute_level_naming_metrics,
    "GATE_J_privacy": None,
}
sub_audits = {
    "axis_canonicalization_parity": AXIS_CANONICALIZATION_PARITY,
    "resize_parity": RESIZE_PARITY,
    "intensity_preprocessing_parity": INTENSITY_PREPROCESSING_PARITY,
    "model_input_shape_parity": MODEL_INPUT_SHAPE_PARITY,
    "prediction_to_mha_coordinate_roundtrip": PREDICTION_TO_MHA_COORDINATE_ROUNDTRIP,
    "mha_geometry_parity": "per_case_in_smoke_test_results" if len(smoke_test_results) else "NOT_RUN",
    "overview_mask_disc_count_consistency": f"{MASK_OVERVIEW_CONSISTENCY_RATE}" if len(mask_overview_consistency) else "NOT_RUN",
    "grading_mask_label_mapping_audit": GRADING_MASK_LABEL_MAPPING_STATUS,
    "relative_disc_ordering_validation": RELATIVE_ORDERING_VALIDATION,
}
gates_df = pd.DataFrame(sorted(gate_status.items()), columns=["gate", "status"])
print(gates_df)
print()
print(json.dumps(sub_audits, indent=2, default=str))


                                     gate                              status
0              GATE_A_checkpoint_identity                                PASS
1                 GATE_B_spider_structure                                PASS
2  GATE_C_public_train_validation_leakage                                PASS
3             GATE_D_smoke_test_execution                                PASS
4         GATE_E_image_mask_exact_pairing                                PASS
5   GATE_F_instance_gt_extraction_metrics                                PASS
6                     GATE_G_fov_analysis                                PASS
7       GATE_H_absolute_anatomical_anchor  UNAVAILABLE_FROM_DATASET_REFERENCE
8    GATE_I_absolute_level_naming_metrics                         UNAVAILABLE
9                          GATE_J_privacy                                None

{
  "axis_canonicalization_parity": "PASS",
  "resize_parity": "PASS",
  "intensity_preprocessing_parity": "PASS",
  "model_input_shape_parit

## Privacy audit (GATE J)


In [44]:
candidate_outputs = {
    "spider_dataset_inventory": spider_dataset_inventory.to_csv(index=False) if len(spider_dataset_inventory) else "",
    "split_inventory": split_inventory.to_csv(index=False) if len(split_inventory) else "",
    "smoke_test_results": smoke_test_results.to_csv(index=False) if len(smoke_test_results) else "",
    "validation_t2_series_inventory": validation_t2_series_inventory.to_csv(index=False) if len(validation_t2_series_inventory) else "",
    "frozen_baseline_case_metrics": frozen_baseline_case_metrics.to_csv(index=False) if len(frozen_baseline_case_metrics) else "",
    "validation_cohort_accounting": json.dumps(validation_cohort_accounting, default=str),
}
privacy_findings = []
for name, text in candidate_outputs.items():
    if "C:\\Users\\" in text or "/Users/" in text:
        privacy_findings.append(f"{name} contains a local filesystem path")

GATE_J_PRIVACY_PASS = len(privacy_findings) == 0
gate_status["GATE_J_privacy"] = "PASS" if GATE_J_PRIVACY_PASS else "FAIL"
gates_df.loc[gates_df["gate"] == "GATE_J_privacy", "status"] = gate_status["GATE_J_privacy"]
print("Privacy findings:", privacy_findings if privacy_findings else "(none)")
print(gates_df)


Privacy findings: (none)
                                     gate                              status
0              GATE_A_checkpoint_identity                                PASS
1                 GATE_B_spider_structure                                PASS
2  GATE_C_public_train_validation_leakage                                PASS
3             GATE_D_smoke_test_execution                                PASS
4         GATE_E_image_mask_exact_pairing                                PASS
5   GATE_F_instance_gt_extraction_metrics                                PASS
6                     GATE_G_fov_analysis                                PASS
7       GATE_H_absolute_anatomical_anchor  UNAVAILABLE_FROM_DATASET_REFERENCE
8    GATE_I_absolute_level_naming_metrics                         UNAVAILABLE
9                          GATE_J_privacy                                PASS


## Decisión y readiness (corregidos)

`ABSOLUTE_LEVEL_ANCHOR_VALIDATED_ON_VALIDATION` **nunca** es un estado posible, porque SPIDER no
ofrece ground truth absoluto en este dataset/run.

**Nota (esta revisión):** `SPIDER_RELATIVE_LOCALIZATION_BASELINE_ESTABLISHED` se basa en los
`execution_gates` (detección, GATE A-F) y puede mantenerse aunque `RELATIVE_ORDERING_VALIDATION`
no sea `PASS` -- son auditorías independientes. La decisión **nunca** afirma que el ordering
relativo está validado; ese estado se reporta por separado en `sub_audits.relative_disc_ordering_validation`.


In [45]:
def overall_gate_status(statuses: list[str]) -> str:
    if any(s == "FAIL" for s in statuses):
        return "FAIL"
    if any(s in ("PARTIAL", "NOT_RUN", "UNAVAILABLE_FROM_DATASET_REFERENCE", "UNAVAILABLE") for s in statuses):
        return "PARTIAL"
    return "PASS"


QUALITY_GATE_OVERALL = overall_gate_status(list(gate_status.values()))

execution_gates = ["GATE_A_checkpoint_identity", "GATE_B_spider_structure", "GATE_C_public_train_validation_leakage",
                    "GATE_D_smoke_test_execution", "GATE_E_image_mask_exact_pairing", "GATE_F_instance_gt_extraction_metrics"]
core_execution_ok = all(gate_status[g] == "PASS" for g in execution_gates)

if core_execution_ok:
    decision = "SPIDER_RELATIVE_LOCALIZATION_BASELINE_ESTABLISHED"
elif any(gate_status[g] not in ("NOT_RUN", "FAIL") for g in execution_gates) and gate_status["GATE_A_checkpoint_identity"] == "PASS":
    decision = "PARTIAL"
else:
    decision = "BLOCKED"

blocking_gates = [g for g, s in gate_status.items() if s not in ("PASS",)]

READY_FOR_AXIAL_CLUSTER_PAIRING = False  # unchanged: no absolute levels exist yet, from any source
READY_FOR_RELATIVE_INSTANCE_VALIDATION_BATCH = bool(GATE_D_smoke_test == "PASS")
READY_FOR_ABSOLUTE_ANCHOR_DATASET_RESEARCH = True  # this notebook establishes the real dataset/evidence needed for that future research, regardless of smoke test outcome

print("QUALITY_GATE_OVERALL:", QUALITY_GATE_OVERALL)
print("Decision:", decision)
print("RELATIVE_ORDERING_VALIDATION (independent sub-audit, does not gate `decision`):", RELATIVE_ORDERING_VALIDATION)
print("Blocking gates:", blocking_gates)
print("ready_for_axial_cluster_pairing:", READY_FOR_AXIAL_CLUSTER_PAIRING)
print("ready_for_relative_instance_validation_batch:", READY_FOR_RELATIVE_INSTANCE_VALIDATION_BATCH)
print("ready_for_absolute_anchor_dataset_research:", READY_FOR_ABSOLUTE_ANCHOR_DATASET_RESEARCH)


QUALITY_GATE_OVERALL: PARTIAL
Decision: SPIDER_RELATIVE_LOCALIZATION_BASELINE_ESTABLISHED
RELATIVE_ORDERING_VALIDATION (independent sub-audit, does not gate `decision`): PASS
Blocking gates: ['GATE_H_absolute_anatomical_anchor', 'GATE_I_absolute_level_naming_metrics']
ready_for_axial_cluster_pairing: False
ready_for_relative_instance_validation_batch: True
ready_for_absolute_anchor_dataset_research: True


## Artefactos — Colab outputs aislados del repo

Runtime outputs (que dependen de datos reales de SPIDER) van a
`<PFI_DRIVE_ROOT>/results|metrics|figures/post_e50/67A/` -- **nunca** a `<PFI_DRIVE_ROOT>/repo`
(ese path de Drive ya no se usa para nada en este notebook -- ni como fuente de código, ni como
destino de escritura). Se genera además un `git_export/` compacto dentro de `results/` para
importar manualmente a la rama local. Los artefactos ya definidos con esquema local
(independientes de datos, ej. quality gates de esta corrida) siguen yendo a
`artifacts/post_e50/spider_level_anchor/` del workspace local (repo local fuera de Colab, workspace
efímero dentro de Colab).


In [46]:
def _json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return str(value)


environment_record = {
    "generated_at": GENERATED_AT, "in_colab": IN_COLAB, "device": str(DEVICE),
    "embedded_runtime_source_commit": EMBEDDED_RUNTIME_SOURCE_COMMIT,
    "embedded_runtime_scope": EMBEDDED_RUNTIME_SCOPE,
    "embedded_runtime_source_files": EMBEDDED_RUNTIME_SOURCE_FILES,
    "dependency_versions": DEPENDENCY_VERSIONS if "DEPENDENCY_VERSIONS" in dir() else None,
    "checkpoint_source": CHECKPOINT_SOURCE, "checkpoint_sha256": checkpoint_sha256,
    "checkpoint_matches_expected": checkpoint_sha256 == EXPECTED_CHECKPOINT_SHA256 if checkpoint_sha256 else None,
    "spider_available": SPIDER_AVAILABLE, "spider_selection_status": SPIDER_SELECTION_STATUS,
    "axis_canonicalization_parity": AXIS_CANONICALIZATION_PARITY,
    "resize_parity": RESIZE_PARITY,
    "intensity_preprocessing_parity": INTENSITY_PREPROCESSING_PARITY,
    "model_input_shape_parity": MODEL_INPUT_SHAPE_PARITY,
    "prediction_to_mha_coordinate_roundtrip": PREDICTION_TO_MHA_COORDINATE_ROUNDTRIP,
}

# Local repo artifacts (schema/gates/summary -- small, always safe to write locally regardless of Colab).
safe_write_text(SPIDER_ANCHOR_DIR / "post_e50_67A_environment.json", json.dumps(environment_record, indent=2, default=_json_default, ensure_ascii=False))
safe_write_text(SPIDER_ANCHOR_DIR / "post_e50_67A_quality_gates.json", json.dumps({"gates": gate_status, "sub_audits": sub_audits, "overall_status": QUALITY_GATE_OVERALL, "warnings": warnings}, indent=2, default=str, ensure_ascii=False))

summary_record = {
    "generated_at": GENERATED_AT, "git_branch": GIT_BRANCH, "git_commit": GIT_COMMIT,
    "spider_available": SPIDER_AVAILABLE, "checkpoint_sha256": checkpoint_sha256,
    "gates": gate_status, "sub_audits": sub_audits, "quality_gate_overall": QUALITY_GATE_OVERALL,
    "decision": decision,
    "ready_for_axial_cluster_pairing": READY_FOR_AXIAL_CLUSTER_PAIRING,
    "ready_for_relative_instance_validation_batch": READY_FOR_RELATIVE_INSTANCE_VALIDATION_BATCH,
    "ready_for_absolute_anchor_dataset_research": READY_FOR_ABSOLUTE_ANCHOR_DATASET_RESEARCH,
    "blocking_gates": blocking_gates,
    "test_split_locked": TEST_SPLIT_LOCKED, "public_test_availability": PUBLIC_TEST_AVAILABILITY, "test_patients_used": test_patients_used,
    "training_performed": TRAINING_PERFORMED, "checkpoint_modified": False,
    "automatic_disc_localization_validated_changed": False,
    "validation_cohort_accounting": validation_cohort_accounting,
    "relative_ordering_validation": RELATIVE_ORDERING_VALIDATION,
    "full_validation_summary": frozen_baseline_summary,
    "warnings": warnings, "limitations": limitations,
}
safe_write_text(SPIDER_ANCHOR_DIR / "post_e50_67A_summary.json", json.dumps(summary_record, indent=2, default=_json_default, ensure_ascii=False))

# Drive runtime outputs (real-data-dependent) -- only if Drive dirs are configured (Colab).
if DRIVE_RESULTS_DIR is not None:
    for target_dir in (DRIVE_RESULTS_DIR, DRIVE_METRICS_DIR, DRIVE_FIGURES_DIR, DRIVE_GIT_EXPORT_DIR):
        target_dir.mkdir(parents=True, exist_ok=True)

    if len(spider_dataset_inventory):
        spider_dataset_inventory.to_csv(DRIVE_RESULTS_DIR / "post_e50_67A_dataset_inventory.csv", index=False)
    if len(validation_t2_series_inventory):
        validation_t2_series_inventory.to_csv(DRIVE_RESULTS_DIR / "validation_t2_series_inventory.csv", index=False)
    if split_leakage_audit_result:
        safe_write_text(DRIVE_RESULTS_DIR / "post_e50_67A_split_leakage_audit.json", json.dumps(split_leakage_audit_result, indent=2, default=_json_default))
    if len(smoke_test_results):
        smoke_test_results.to_csv(DRIVE_METRICS_DIR / "post_e50_67A_smoke_test_results.csv", index=False)
    if len(mask_overview_consistency):
        mask_overview_consistency.to_csv(DRIVE_RESULTS_DIR / "mask_overview_consistency.csv", index=False)
    if len(grading_mask_label_mapping_audit):
        grading_mask_label_mapping_audit.to_csv(DRIVE_RESULTS_DIR / "grading_mask_label_mapping_audit.csv", index=False)
    if len(frozen_baseline_case_metrics):
        frozen_baseline_case_metrics.to_csv(DRIVE_METRICS_DIR / "post_e50_67A_frozen_baseline_case_metrics.csv", index=False)
    if frozen_baseline_summary:
        safe_write_text(DRIVE_METRICS_DIR / "post_e50_67A_frozen_baseline_summary.json", json.dumps(frozen_baseline_summary, indent=2, default=_json_default))
    pd.DataFrame(anchor_experiment_rows).to_csv(DRIVE_METRICS_DIR / "post_e50_67A_anchor_experiment_metrics.csv", index=False)

    # Compact git_export -- small, schema-clean files intended for manual import into the repo branch.
    safe_write_text(DRIVE_GIT_EXPORT_DIR / "post_e50_67A_summary.json", json.dumps(summary_record, indent=2, default=_json_default, ensure_ascii=False))
    safe_write_text(DRIVE_GIT_EXPORT_DIR / "post_e50_67A_quality_gates.json", json.dumps({"gates": gate_status, "sub_audits": sub_audits}, indent=2, default=str, ensure_ascii=False))
    if len(smoke_test_results):
        smoke_test_results.to_csv(DRIVE_GIT_EXPORT_DIR / "post_e50_67A_smoke_test_results.csv", index=False)
    print("Drive runtime outputs written under:", "results/metrics/figures/post_e50/67A (relative to PFI_DRIVE_ROOT)")
    print("Compact git_export written under: results/post_e50/67A/git_export/")
else:
    warnings.append("Drive output directories not configured in this run (not in Colab / PFI_DRIVE_ROOT unset) -- only local repo-scoped artifacts were written.")

print("Local repo artifacts written under artifacts/post_e50/spider_level_anchor/:")
for f in sorted(SPIDER_ANCHOR_DIR.rglob("*")):
    if f.is_file():
        print(" -", f.relative_to(REPO_ROOT))


Drive runtime outputs written under: results/metrics/figures/post_e50/67A (relative to PFI_DRIVE_ROOT)
Compact git_export written under: results/post_e50/67A/git_export/
Local repo artifacts written under artifacts/post_e50/spider_level_anchor/:
 - artifacts/post_e50/spider_level_anchor/post_e50_67A_environment.json
 - artifacts/post_e50/spider_level_anchor/post_e50_67A_quality_gates.json
 - artifacts/post_e50/spider_level_anchor/post_e50_67A_summary.json


## Limitaciones (agregado)


In [47]:
limitations.extend([
    "SPIDER instance labels (vertebrae 1,2,3...; canal 100; discs 201,202,203...) are RELATIVE, "
    "bottom-up identity, not absolute anatomical levels -- L1-L2...L5-S1 naming is UNAVAILABLE_FROM_SPIDER_REFERENCE.",
    "The public SPIDER split has no test subset locally; the hidden challenge test is out of scope and never accessed.",
    f"The checkpoint resolver never trusts a candidate by path/filename alone -- it always verifies by computed SHA-256 (expected: {EXPECTED_CHECKPOINT_SHA256}). "
    "The LOCAL git checkout's models/final/sagittal_spider_multiclass_final_best.pt DOES match this expected SHA (confirmed: cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944) "
    "and is accepted when resolved outside Colab. In Colab, PFI_POST_E50_SAGITTAL_CHECKPOINT is preferred and accepted only on a SHA match; "
    "a stale Drive-only copy with a different SHA (7dd393cc750311c98003516d8110136310c31e8b6f0f00b6815f949fd61ef15b) was observed in one prior Colab session and is rejected by design, but that staleness was specific to that Drive clone, not the local git history.",
    "MHA slice-axis handling reproduces notebooks/45_gcs_spider_final_training.ipynb's shape heuristic + fixed axis=2 exactly (code-identity parity), not an independently re-derived or spacing-verified convention.",
    "GRADING_MASK_LABEL_MAPPING_STATUS, if VALIDATED, is scoped to the cases actually processed in this run (smoke test / full validation), not the full 218-patient cohort unless RUN_FULL_VALIDATION=True was executed.",
    "GATE D (smoke test) PASS means 3/3 cases executed without runtime failure -- it does NOT imply high detection accuracy; no performance threshold is defined.",
    "No training was performed; no checkpoint was modified; AUTOMATIC_DISC_LOCALIZATION_VALIDATED was not touched; Notebook 67 was not modified.",
    "TEST_SPLIT_LOCKED=True structurally prevents any test execution in this notebook.",
])
for w in warnings:
    if w not in limitations:
        limitations.append(w)
for item in limitations:
    print("-", item)


- SPIDER instance labels (vertebrae 1,2,3...; canal 100; discs 201,202,203...) are RELATIVE, bottom-up identity, not absolute anatomical levels -- L1-L2...L5-S1 naming is UNAVAILABLE_FROM_SPIDER_REFERENCE.
- The public SPIDER split has no test subset locally; the hidden challenge test is out of scope and never accessed.
- The checkpoint resolver never trusts a candidate by path/filename alone -- it always verifies by computed SHA-256 (expected: cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944). The LOCAL git checkout's models/final/sagittal_spider_multiclass_final_best.pt DOES match this expected SHA (confirmed: cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944) and is accepted when resolved outside Colab. In Colab, PFI_POST_E50_SAGITTAL_CHECKPOINT is preferred and accepted only on a SHA match; a stale Drive-only copy with a different SHA (7dd393cc750311c98003516d8110136310c31e8b6f0f00b6815f949fd61ef15b) was observed in one prior Colab session and is reje

## Reporte (Markdown) — corregido: sin afirmaciones Stage-A obsoletas


In [48]:
report_lines = []
report_lines.append("# Post-E50 SPIDER Level Anchor Validation")
report_lines.append("")
report_lines.append("## Objective")
report_lines.append("")
report_lines.append(
    "Validate the Notebook 67 sagittal instance-localization pipeline against real SPIDER data. "
    "SPIDER provides RELATIVE instance identity (bottom-up numbering), not absolute anatomical "
    "levels -- this notebook never claims L1-L2...L5-S1 ground truth from SPIDER."
)
report_lines.append("")
report_lines.append("## Execution status (this run)")
report_lines.append("")
if SPIDER_AVAILABLE:
    report_lines.append(f"Stage B: SPIDER data was accessed in this run. GATE A (checkpoint identity): `{gate_status['GATE_A_checkpoint_identity']}`.")
    if gate_status["GATE_A_checkpoint_identity"] != "PASS":
        report_lines.append("**Checkpoint identity was NOT verified in this run** -- no claim of 'checkpoint verified' is made.")
else:
    report_lines.append("Stage A / local validation: SPIDER data was not accessed in this run (expected outside Colab).")
report_lines.append("")
report_lines.append("## Embedded runtime provenance")
report_lines.append("")
report_lines.append(f"- EMBEDDED_RUNTIME_SOURCE_COMMIT: `{EMBEDDED_RUNTIME_SOURCE_COMMIT}`")
report_lines.append(f"- EMBEDDED_RUNTIME_SCOPE: `{EMBEDDED_RUNTIME_SCOPE}`")
report_lines.append(
    "- This notebook embeds a minimal frozen copy of `build_checkpoint_model`, `resize_image`, "
    "`robust_percentile_normalize`, `connected_instances`, and `MODEL_REGISTRY` from the immutable "
    "Notebook-67 commit above -- it is NOT a new implementation, and it is NOT imported from "
    "`ai_service.pfi_ai_service.*` at runtime (no `PFI_AI_REPO_ROOT`, no `<PFI_DRIVE_ROOT>/repo` dependency)."
)
report_lines.append("")
report_lines.append("## Checkpoint resolver")
report_lines.append("")
report_lines.append(f"- Expected SHA-256: `{EXPECTED_CHECKPOINT_SHA256}`")
report_lines.append(f"- Resolved source: `{CHECKPOINT_SOURCE}`, sha256: `{checkpoint_sha256}`")
report_lines.append(f"- GATE A: `{gate_status['GATE_A_checkpoint_identity']}`")
report_lines.append(
    "- The resolver never trusts a candidate by path/filename -- it always verifies by computed SHA-256. "
    "The LOCAL git checkout's models/final/sagittal_spider_multiclass_final_best.pt matches the expected SHA "
    "and is accepted outside Colab; in Colab, PFI_POST_E50_SAGITTAL_CHECKPOINT is preferred and accepted only on a SHA match "
    "(a previously observed Drive-only copy with a mismatched SHA is rejected by design, not the local git checkout)."
)
report_lines.append("")
report_lines.append("## Preprocessing parity (4 independent sub-audits -- do not overclaim from axis parity alone)")
report_lines.append("")
report_lines.append(f"- AXIS_CANONICALIZATION_PARITY: `{AXIS_CANONICALIZATION_PARITY}` "
                     "(code-identity with notebooks/45_gcs_spider_final_training.ipynb: same shape heuristic, same fixed axis=2)")
report_lines.append(f"- RESIZE_PARITY: `{RESIZE_PARITY}` (source: ai_service.pfi_ai_service.real_inference_runtime.resize_image, embedded verbatim in STEP 1B -- commit {EMBEDDED_RUNTIME_SOURCE_COMMIT}, not imported)")
report_lines.append(f"- INTENSITY_PREPROCESSING_PARITY: `{INTENSITY_PREPROCESSING_PARITY}` (source: ai_service.pfi_ai_service.real_inference_runtime.robust_percentile_normalize, embedded verbatim in STEP 1B -- commit {EMBEDDED_RUNTIME_SOURCE_COMMIT}, not imported)")
report_lines.append(f"- MODEL_INPUT_SHAPE_PARITY: `{MODEL_INPUT_SHAPE_PARITY}` (sagittal_runtime_meta['targetSize'] vs training TARGET_SIZE=(256,256))")
report_lines.append("")
report_lines.append("## Prediction -> original MHA coordinate roundtrip (critical, gates the smoke test)")
report_lines.append("")
report_lines.append(f"- PREDICTION_TO_MHA_COORDINATE_ROUNDTRIP: `{PREDICTION_TO_MHA_COORDINATE_ROUNDTRIP}`")
report_lines.append(
    "- Validated via synthetic self-tests: (1) canonical index <-> native SimpleITK index mapping round-trip for both the "
    "swap and no-swap cases of canonicalize_spider_array; (2) SimpleITK TransformContinuousIndexToPhysicalPoint <-> "
    "TransformPhysicalPointToContinuousIndex round-trip on a synthetic image; (3) native-pixel <-> model-grid resize-inverse "
    "round-trip. At runtime, each predicted centroid is additionally self-checked to round-trip back to its native canonical "
    "index via image_sitk (the ORIGINAL, pre-preprocessing image) -- both predicted and GT centroids end in the same physical "
    "space by construction (both call canonical_index_to_physical_xyz against their own image_sitk / mask_sitk objects). "
    "If this gate is not PASS, run_v67_frozen_pipeline_on_case() returns status=coordinate_roundtrip_not_validated and no "
    "centroid_error_mm is produced for that case."
)
report_lines.append("")
report_lines.append("## Dataset verification / splits / leakage")
report_lines.append("")
report_lines.append(f"- GATE B (dataset structure): `{gate_status['GATE_B_spider_structure']}`")
report_lines.append(f"- GATE C (train/validation leakage): `{gate_status['GATE_C_public_train_validation_leakage']}`")
report_lines.append(f"- GATE E (image-mask exact pairing): `{gate_status['GATE_E_image_mask_exact_pairing']}`")
report_lines.append(f"- public_test_availability: `{PUBLIC_TEST_AVAILABILITY}`")
report_lines.append("")
report_lines.append(json.dumps(split_leakage_audit_result, indent=2, default=str) if split_leakage_audit_result else "_Not computed in this run._")
report_lines.append("")
report_lines.append("## Validation cohort accounting")
report_lines.append("")
report_lines.append(
    f"- validation_patients_total: `{validation_cohort_accounting['validation_patients_total']}`, "
    f"validation_patients_t2_evaluable: `{validation_cohort_accounting['validation_patients_t2_evaluable']}`, "
    f"validation_patients_not_evaluable: `{validation_cohort_accounting['validation_patients_not_evaluable']}`"
)
report_lines.append(f"- not_evaluable_reasons: `{validation_cohort_accounting.get('not_evaluable_reasons', {})}`")
report_lines.append("- A full validation batch attempts exactly the T2-evaluable count, never the total validation patient count.")
report_lines.append("")
report_lines.append("## Smoke test (real pipeline execution)")
report_lines.append("")
report_lines.append(f"- GATE D (execution): `{gate_status['GATE_D_smoke_test_execution']}` -- PASS means 3/3 cases ran without runtime failure, NOT high accuracy.")
report_lines.append(f"- GATE F (instance GT extraction + metrics): `{gate_status['GATE_F_instance_gt_extraction_metrics']}`")
report_lines.append("")
report_lines.append(smoke_test_results.to_markdown(index=False) if len(smoke_test_results) else "_Not run in this session (no SPIDER data)._")
report_lines.append("")
report_lines.append("## Relative disc ordering validation (separate from GATE F detection)")
report_lines.append("")
report_lines.append(f"- RELATIVE_ORDERING_VALIDATION: `{RELATIVE_ORDERING_VALIDATION}`")
report_lines.append(
    "- Computed per matched pair (Hungarian output) as predicted_rank vs gt_relative_instance_label. "
    "relative_order_direction is NEVER derived from GT -- it stays `ORDER_DIRECTION_UNRESOLVED` unless independent "
    "physical-space/anatomical evidence resolves it (not available for SPIDER MHA in this notebook). "
    "See the sign-ambiguity investigation above (spine_axis_from_points self-test) for the confirmed root cause "
    "of pred_index-to-gt_index direction differing across cases."
)
report_lines.append("")
report_lines.append("## Mask/overview/grading audits")
report_lines.append("")
report_lines.append(f"- overview-mask disc-count consistency rate: `{sub_audits['overview_mask_disc_count_consistency']}`")
report_lines.append(f"- GRADING_MASK_LABEL_MAPPING_STATUS: `{GRADING_MASK_LABEL_MAPPING_STATUS}` (never converted to an absolute anatomical level)")
report_lines.append("")
report_lines.append("## FOV analysis")
report_lines.append("")
report_lines.append(f"- GATE G: `{gate_status['GATE_G_fov_analysis']}`. Notebook 67's real-DICOM result of 7 candidates is NOT concluded to be an error by this analysis.")
report_lines.append(
    "- `validation_series_num_discs_distribution` (ALL validation series, may include >1 per patient) is now named "
    "explicitly to avoid implying it is per-patient. `validation_primary_t2_num_discs_distribution` (exactly one "
    "primary T2 per T2-evaluable patient) is the primary comparison distribution against this pipeline."
)
report_lines.append("")
report_lines.append("## Absolute anatomical anchor")
report_lines.append("")
report_lines.append(f"- GATE H: `{gate_status['GATE_H_absolute_anatomical_anchor']}` -- a valid methodological outcome, not a failure.")
report_lines.append(f"- GATE I (absolute level naming metrics): `{gate_status['GATE_I_absolute_level_naming_metrics']}` -- no L1-L2...L5-S1 confusion matrix generated (no ground truth exists).")
report_lines.append(pd.DataFrame(anchor_experiment_rows).to_markdown(index=False))
report_lines.append("")
report_lines.append("## Quality gates")
report_lines.append("")
report_lines.append(gates_df.to_markdown(index=False))
report_lines.append("")
report_lines.append("## Results")
report_lines.append("")
report_lines.append(f"- Overall quality gate: `{QUALITY_GATE_OVERALL}`")
report_lines.append(f"- Decision: `{decision}`")
report_lines.append(f"- ready_for_axial_cluster_pairing: `{READY_FOR_AXIAL_CLUSTER_PAIRING}`")
report_lines.append(f"- ready_for_relative_instance_validation_batch: `{READY_FOR_RELATIVE_INSTANCE_VALIDATION_BATCH}`")
report_lines.append(f"- ready_for_absolute_anchor_dataset_research: `{READY_FOR_ABSOLUTE_ANCHOR_DATASET_RESEARCH}`")
report_lines.append("")
report_lines.append("## Test lock")
report_lines.append("")
report_lines.append(f"- TEST_SPLIT_LOCKED: `{TEST_SPLIT_LOCKED}` (structurally enforced) · test_patients_used: `{test_patients_used}` · public_test_availability: `{PUBLIC_TEST_AVAILABILITY}`")
report_lines.append("")
report_lines.append("## Limitations")
report_lines.append("")
for item in limitations:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## What 67A proves (this run)")
report_lines.append("")
if gate_status["GATE_D_smoke_test_execution"] == "PASS":
    report_lines.append("> The V67_FROZEN_BASELINE pipeline executed end-to-end on real SPIDER MHA volumes with the "
                         "SHA-verified checkpoint: MHA loading with training-parity axis handling, frozen inference, "
                         "disc-instance extraction, multi-slice consensus, physical-space Hungarian matching against "
                         "real mask-derived ground truth, and relative-instance metrics -- all without absolute level claims.")
else:
    report_lines.append("> No real SPIDER execution evidence exists in this run (Stage A / local validation only, or "
                         "the smoke test did not PASS). No performance or execution claim is made beyond what actually ran.")
report_lines.append("")
report_lines.append("## What 67A does not prove (this run)")
report_lines.append("")
for item in [
    "no absolute anatomical level ground truth exists in SPIDER for this dataset/run -- L1-L2...L5-S1 is never claimed",
    "GATE D PASS is execution-only, not a performance/accuracy claim",
    f"no full validation batch (would attempt exactly {validation_cohort_accounting['validation_patients_t2_evaluable']} "
    f"T2-evaluable patients, not the {validation_cohort_accounting['validation_patients_total']} total validation patients) "
    "unless RUN_FULL_VALIDATION was explicitly set True and executed",
    "relative disc ordering is NOT claimed as validated beyond the RELATIVE_ORDERING_VALIDATION sub-audit's actual scope "
    "(smoke-test cohort only, unless full validation was run) -- ORDER_DIRECTION_UNRESOLVED unless independent physical evidence exists",
    "test split was never touched",
    "Notebook 67 was not modified",
]:
    report_lines.append(f"- {item}")
report_lines.append("")

report_text = "\n".join(report_lines)
safe_write_text(REPORT_DIR / "post_e50_spider_level_anchor_validation_report.md", report_text)
print(f"Report written: {(REPORT_DIR / 'post_e50_spider_level_anchor_validation_report.md').relative_to(REPO_ROOT)}")


Report written: reports/post_e50/post_e50_spider_level_anchor_validation_report.md


## EXPERIMENT STATUS


In [49]:
EXECUTION_SECONDS = time.time() - EXECUTION_START
gpu_label = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)"

status_block = f'''EXPERIMENT STATUS

Experiment:
Post-E50 SPIDER Level Anchor Validation

Training performed:
NO

Frozen checkpoint:
{checkpoint_sha256}

Environment:
{"COLAB" if IN_COLAB else "LOCAL"}

GPU:
{gpu_label}

SPIDER root:
{"CONFIGURED" if SPIDER_AVAILABLE else "MISSING"}

Dataset verification:
{gate_status["GATE_B_spider_structure"]}

Validation patients total:
{validation_cohort_accounting['validation_patients_total']}

Validation patients T2-evaluable:
{validation_cohort_accounting['validation_patients_t2_evaluable']}

Validation patients not evaluable:
{validation_cohort_accounting['validation_patients_not_evaluable']}

Test patients used:
{test_patients_used}

Leakage audit:
{gate_status["GATE_C_public_train_validation_leakage"]}

Image-mask pairing:
{gate_status["GATE_E_image_mask_exact_pairing"]}

Smoke test:
{gate_status["GATE_D_smoke_test_execution"]}

Relative ordering validation:
{RELATIVE_ORDERING_VALIDATION}

Full validation:
{"NOT_RUN" if not RUN_FULL_VALIDATION else ("PASS" if frozen_baseline_summary else "FAIL")}

Disc instance GT metrics:
{"AVAILABLE" if gate_status["GATE_F_instance_gt_extraction_metrics"] == "PASS" else "UNAVAILABLE"}

FOV analysis:
{gate_status["GATE_G_fov_analysis"]}

Absolute anchor:
{gate_status["GATE_H_absolute_anatomical_anchor"]}

Level metrics:
{gate_status["GATE_I_absolute_level_naming_metrics"]}

Test split:
LOCKED

Public test availability:
{PUBLIC_TEST_AVAILABILITY}

Privacy:
{gate_status["GATE_J_privacy"]}

Overall quality gate:
{QUALITY_GATE_OVERALL}

Decision:
{decision}

Ready for axial cluster pairing:
{"YES" if READY_FOR_AXIAL_CLUSTER_PAIRING else "NO"}

Ready for relative instance validation batch:
{"YES" if READY_FOR_RELATIVE_INSTANCE_VALIDATION_BATCH else "NO"}

Ready for absolute anchor dataset research:
{"YES" if READY_FOR_ABSOLUTE_ANCHOR_DATASET_RESEARCH else "NO"}

Blocking gates:
{blocking_gates if blocking_gates else "[]"}

Execution seconds:
{EXECUTION_SECONDS:.1f}

Warnings:
{chr(10).join(warnings) if warnings else "(none)"}
'''

print(status_block)


EXPERIMENT STATUS

Experiment:
Post-E50 SPIDER Level Anchor Validation

Training performed:
NO

Frozen checkpoint:
cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944

Environment:
COLAB

GPU:
none (CPU)

SPIDER root:
CONFIGURED

Dataset verification:
PASS

Validation patients total:
39

Validation patients T2-evaluable:
38

Validation patients not evaluable:
1

Test patients used:
0

Leakage audit:
PASS

Image-mask pairing:
PASS

Smoke test:
PASS

Relative ordering validation:
PASS

Full validation:
PASS

Disc instance GT metrics:
AVAILABLE

FOV analysis:
PASS

Absolute anchor:
UNAVAILABLE_FROM_DATASET_REFERENCE

Level metrics:
UNAVAILABLE

Test split:
LOCKED

Public test availability:
HIDDEN_EXTERNAL_NOT_AVAILABLE

Privacy:
PASS

Overall quality gate:
PARTIAL

Decision:
SPIDER_RELATIVE_LOCALIZATION_BASELINE_ESTABLISHED

Ready for axial cluster pairing:
NO

Ready for relative instance validation batch:
YES

Ready for absolute anchor dataset research:
YES

Blocking gates:
